# SuReSuite 04 · Component inventory and disruptions

**You will:** load your project's real data and the policies as they are on /policies now,
run the baseline on your own machine, trace every material's stock week by week, knock out
each supplier in turn, and follow the worst one down to the materials and products it hits.
**Time:** about 20 minutes (longer on a large project: step 5 runs one simulation per target).
**You need:** nothing for demo mode; with a key, `read:data` and `read:policies` (`read:runs`
to use your saved scenario). Every user can create that key on /developer. `write:data` and
`write:policies` (admins and modelers) let the notebook freeze your data and save your policies
first, so it never has to work from an older copy.

Nothing here changes your project. It runs on **this machine**, so it costs no quota.

**Two limits:** the material traces in steps 4 and 6 are **one seed** (the engine keeps
per-material weeks only for a single-replication run); the KPIs in steps 3 and 5 average over
every replication. A sub-assembly has no stock line of its own: you trace its components.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
# The app fills in PROJECT_ID when you download this notebook from /developer →
# Notebook; choose SCENARIO_ID and POLICY_VERSION_ID yourself (that tab lists them)
# or leave them blank. Leave it all as it is to try the offline demo first.
BASE_URL = "https://wckdrutwkytwcomrlpib.supabase.co/functions/v1/api/v1"
PROJECT_ID = ""          # "" = the first project your key can see
SCENARIO_ID = ""         # a baseline scenario; "" = pick one without disruptions (or create one)
POLICY_VERSION_ID = ""   # a policy snapshot; "" = snapshot the current configuration
SUPPLIER_ID = ""         # a supplier id from your own data, for the disruption examples
MODE = "auto"            # "auto" = live when an API key is found, else the offline demo; "live"; "demo"
ACKNOWLEDGE_WARNINGS = False  # True = run anyway when the pre-run data check only warns (missing values use the platform defaults)

In [ ]:
# @title Setup: packages and API key
# ── SETUP: packages and API key ─────────────────────────────────────────────
# Installs only what is missing (Google Colab already has all of it), then finds
# your API key: the Colab Secrets panel first, then the SURESUITE_API_KEY
# environment variable. MODE = "live" also asks for it with a hidden prompt.
# The key is never printed and never stored in the notebook.
import importlib.util
import os
import subprocess
import sys

_needed = ["requests", "pandas", "matplotlib", "pyarrow", "openpyxl"]
_missing = [p for p in _needed if importlib.util.find_spec(p) is None]
if _missing:
    print("installing:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

API_KEY = None
if MODE != "demo":
    try:
        from google.colab import userdata  # Colab: the Secrets panel (key icon, left sidebar)
        API_KEY = userdata.get("SURESUITE_API_KEY")
    except Exception:
        pass
    API_KEY = API_KEY or os.environ.get("SURESUITE_API_KEY")
    if not API_KEY and MODE == "live":
        from getpass import getpass
        API_KEY = getpass("SuReSuite API key (sk_test_… or sk_live_…): ").strip()
    if API_KEY and not API_KEY.startswith("sk_"):
        raise ValueError("an API key starts with sk_test_ or sk_live_ — create one on the app's /developer page")
    if MODE == "live" and not API_KEY:
        raise ValueError("MODE is \"live\" but no API key was found")

LIVE = bool(API_KEY)
if LIVE:
    print(f"LIVE mode — {API_KEY.split('_')[1]} key, talking to {BASE_URL}")
else:
    print("DEMO mode — no API key found, so this notebook replays recorded engine output for the\n"
          "Example project (1 product, 2 materials, 3 suppliers). Every cell runs the same way in\n"
          "live mode. To go live, add SURESUITE_API_KEY (see the Setup notes) and run all cells again.")

In [ ]:
# @title SuReSuite notebook library (run it; no need to read it)
# ── SuReSuite notebook library ──────────────────────────────────────────────
# The `suresuite` Python package's client (python/suresuite/client.py), inlined
# into each notebook by scripts/notebooks/build-notebooks.mjs. Run this cell;
# you do not need to read it. It defines:
#   connect()                      the API client (live, or the offline demo), checked
#                                  against the scopes the notebook needs
#   api.run_and_wait(...)          dispatch a simulation run and poll it to the end
#   kpi_table / replications_frame / paired_compare / assert_disruption_applied
#   editing_policies(...)          edit policies and ALWAYS put them back
#   plot_kpis / plot_series        small, honest charts
# Every helper talks to the public /v1 API the same way in live and demo mode.
import base64
import contextlib
import copy
import hashlib
import io
import json
import math
import re
import time
import uuid
import zlib

import pandas as pd
import requests

try:  # the charts need matplotlib; the package works without it (`pip install suresuite[plots]`)
    import matplotlib.pyplot as plt
except ImportError:  # pragma: no cover - exercised only without the extra
    plt = None

try:  # as the `suresuite` package; in a notebook the KPI_DISPLAY cell is inlined below instead
    from .kpi_display import KPI_DISPLAY
except ImportError:
    pass

FAMILIES = ("sourcing", "inventory", "transport", "fulfillment", "production", "recovery", "demand")
TERMINAL = ("done", "failed", "cancelled")  # a run's lifecycle: queued → running → done | failed | cancelled
PALETTE = ["#2a78d6", "#eda100", "#008300", "#e87ba4", "#7a5af8", "#5f6b7a"]  # fixed order per series
T95 = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447, 7: 2.365, 8: 2.306, 9: 2.262,
       10: 2.228, 11: 2.201, 12: 2.179, 13: 2.16, 14: 2.145, 15: 2.131, 16: 2.12, 17: 2.11,
       18: 2.101, 19: 2.093, 20: 2.086, 21: 2.08, 22: 2.074, 23: 2.069, 24: 2.064, 25: 2.06,
       26: 2.056, 27: 2.052, 28: 2.048, 29: 2.045, 30: 2.042}
_MONEY = {"symbol": "€"}  # replaced by the engine's declared unit when the catalog is read
KEY_FIELDS = ("key_prefix", "env", "principal", "scopes", "project_ids")  # GET /key, as the gateway answers it


class SuReSuiteError(RuntimeError):
    """The API's error envelope {"error": {"code", "message", "details"}} as an exception."""

    def __init__(self, status, code, message, details=None, request_id=None):
        hint = ""
        if code == "missing_scope":  # a key's scopes are fixed when it is created
            hint = (" — create a key that has this scope on the app's /developer page"
                    " (Create API key), then set SURESUITE_API_KEY to it and run all cells again")
        super().__init__(f"[{status} {code}] {message}{hint}")
        self.status, self.code, self.message = status, code, message
        self.details, self.request_id = details, request_id


class Stop(Exception):
    """A notebook cannot go on, and says why and what to do. In Jupyter and Colab it
    shows as that one message rather than a traceback (`%tb` still has the trace)."""

    def _render_traceback_(self):
        return [f"Stopped: {self}"]


# ── transports: how a request reaches the platform ──────────────────────────
class HttpTransport:
    """The real thing: HTTPS to the /v1 gateway with your key."""

    def __init__(self, base_url, api_key, timeout=60):
        self.base = base_url.rstrip("/")
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers["Authorization"] = f"Bearer {api_key}"

    def request(self, method, path, params=None, body=None, headers=None):
        r = self.session.request(method, self.base + path, params=params, json=body,
                                 headers=headers or {}, timeout=self.timeout)
        try:
            payload = r.json() if r.content else {}
        except ValueError:  # an HTML error page from a proxy, say
            payload = {"error": {"code": "non_json_response", "message": r.text[:200]}}
        return r.status_code, dict(r.headers), payload

    def fetch(self, url):
        # A signed storage URL carries its own authorization: never send the API key to it.
        r = requests.get(url, timeout=self.timeout)
        r.raise_for_status()
        return r.content


class SuReSuite:
    """Client for the SuReSuite public /v1 API."""

    def __init__(self, transport, live):
        self.t, self.live = transport, live
        self._catalog = {}

    # ── plumbing ──
    def _call(self, method, path, params=None, body=None, headers=None, retries=4):
        for attempt in range(retries + 1):
            try:
                status, hdrs, payload = self.t.request(method, path, params=params, body=body, headers=headers)
            except requests.RequestException as e:  # network trouble: retry with backoff, then give up
                if attempt == retries:
                    raise
                wait = 2 ** (attempt + 1)
                print(f"  network error ({type(e).__name__}) — retrying in {wait}s")
                time.sleep(wait)
                continue
            err = (payload or {}).get("error") or {} if status >= 400 else {}
            if status == 429 and err.get("code") in ("rate_limited", "too_many_failed_auths") and attempt < retries:
                wait = min(int(hdrs.get("Retry-After", "5") or 5), 60)
                print(f"  rate-limited — waiting {wait}s")
                time.sleep(wait)
                continue
            if status in (502, 503, 504) and attempt < retries:
                time.sleep(2 ** (attempt + 1))
                continue
            remaining = hdrs.get("X-RateLimit-Remaining")
            if remaining is not None and remaining.isdigit() and int(remaining) <= 5:
                print(f"  note: {remaining} requests left in this key's rate-limit window")
            if status >= 400:
                raise SuReSuiteError(status, err.get("code", "unknown"), err.get("message", ""),
                                     err.get("details"), hdrs.get("X-Request-Id"))
            return payload
        raise RuntimeError("unreachable")

    def get(self, path, **params):
        return self._call("GET", path, params={k: v for k, v in params.items() if v is not None} or None)

    def post(self, path, body=None, idempotency_key=None):
        headers = {"Idempotency-Key": idempotency_key} if idempotency_key else None
        return self._call("POST", path, body=body or {}, headers=headers)

    def put(self, path, body):
        return self._call("PUT", path, body=body)

    def list_all(self, path, limit=100, **params):
        """Follow cursor pagination (?limit=&cursor=) until the last page."""
        rows, cursor = [], None
        while True:
            page = self.get(path, limit=limit, cursor=cursor, **params)
            rows += page["data"]
            cursor = page.get("next_cursor")
            if not cursor:
                return rows

    # ── the key itself ──
    def key(self):
        """The calling key as the platform sees it now: `scopes`, `env`, `principal`
        and `project_ids`. None from a gateway that predates `GET /key`."""
        try:
            return self.get("/key")
        except SuReSuiteError as e:
            if e.code == "route_not_found":
                return None
            raise

    # ── projects and input data ──
    def projects(self):
        return self.list_all("/projects")

    def project(self, project_id):
        return self.get(f"/projects/{project_id}")

    def dataset_versions(self, project_id):
        return self.get(f"/projects/{project_id}/dataset-versions")["data"]

    def freeze_dataset(self, project_id, label=None):
        return self.post(f"/projects/{project_id}/datasets:freeze", {"label": label} if label else {})

    # ── policies ──
    def catalog(self, project_id):
        if project_id not in self._catalog:
            cat = self.get(f"/projects/{project_id}/policy-catalog")
            money = {k["unit"] for k in cat.get("kpis") or [] if k.get("name") in ("revenue", "lost_sales_value")}
            if len(money) == 1:
                _MONEY["symbol"] = money.pop()
            self._catalog[project_id] = cat
        return self._catalog[project_id]

    def policies(self, project_id):
        return self.get(f"/projects/{project_id}/policies")

    def policy_defaults(self, project_id):
        """The seven policy families, each a dict ({} = engine defaults)."""
        d = self.policies(project_id)["defaults"] or {}
        return {f: dict(d.get(f) or {}) for f in FAMILIES}

    def set_policy(self, project_id, family, **fields):
        """Change some fields of ONE family. PUT replaces a family as a whole, so this
        reads the family, merges your fields in, and writes the whole family back."""
        if family not in FAMILIES:
            raise ValueError(f"family must be one of {FAMILIES}")
        merged = {**self.policy_defaults(project_id)[family], **fields}
        self.put(f"/projects/{project_id}/policies", {"defaults": {family: merged}})
        return merged

    def snapshot_policies(self, project_id, label):
        """An immutable policy version. An unchanged configuration returns the version
        that already holds it (content-addressed), so the label may be an older one."""
        return self.post(f"/projects/{project_id}/policy-versions", {"label": label})

    def dataset_version(self, project_id, version="latest", tables=None):
        """One frozen dataset version WITH its rows (`snapshot`): exactly what a run
        reads. `version` is an id or "latest"; `tables` narrows it."""
        return self.get(f"/projects/{project_id}/dataset-versions/{version}",
                        tables=",".join(tables) if tables else None)

    def policy_version(self, project_id, version="latest"):
        """One frozen policy version with its `snapshot` and `policy_hash`."""
        return self.get(f"/projects/{project_id}/policy-versions/{version}")

    def engine(self, version=None):
        """The engine's wheels: sha256, size and a short-lived signed URL each.

        ``version`` — ``"0.4.0"`` (the newest build of that version) or an exact
        build ``"scsim-0.4.0+<digest>"`` — fetches an EARLIER engine, so a stored
        result can be re-run on the engine that produced it (PLAN.md §25 WP 15.3)."""
        return self.get("/engine", version=version)

    def policy_versions(self, project_id):
        return self.get(f"/projects/{project_id}/policy-versions")["data"]

    # ── scenarios ──
    def scenarios(self, project_id):
        return self.list_all(f"/projects/{project_id}/scenarios")

    def create_scenario(self, project_id, name, **fields):
        return self.post(f"/projects/{project_id}/scenarios", {"name": name, **fields})

    # ── runs ──
    def run(self, run_id):
        return self.get(f"/runs/{run_id}")

    def replications(self, run_id):
        return self.list_all(f"/runs/{run_id}/replications")

    def validation(self, run_id):
        return self.get(f"/runs/{run_id}/validation")

    def cancel(self, run_id):
        return self.post(f"/runs/{run_id}:cancel")

    def series(self, run_id, expired_ok=False):
        """Every weekly series of a run as ONE long table:
        rep_index · model_rep · event_rep · week · one column per series.
        `expired_ok=True` returns None (and says so) for a run whose series have expired."""
        page = self.get(f"/runs/{run_id}/replications", include="time_series", limit=100)
        where = page.get("series") or {}
        if where.get("location") == "object" and where.get("url"):
            blob = self.t.fetch(where["url"])  # live: Parquet bytes; demo: a column table
            return pd.read_parquet(io.BytesIO(blob)) if isinstance(blob, (bytes, bytearray)) else pd.DataFrame(blob)
        if where.get("location") == "expired":
            if expired_ok:
                print(f"run {run_id}: its weekly series have expired (the KPIs above are kept) — dispatch it again "
                      "with force_rerun=True to compute them")
                return None
            raise SuReSuiteError(410, "series_expired",
                                 "this run's weekly series have expired; re-run its RunKey to reproduce them",
                                 where)
        rows = []  # an older run keeps its series on each replication row
        for rep in self.list_all(f"/runs/{run_id}/replications", include="time_series"):
            ts = rep.get("time_series") or {}
            for week in range(max((len(v) for v in ts.values()), default=0)):
                rows.append({"rep_index": rep["rep_index"], "model_rep": (rep.get("kpis") or {}).get("model_rep"),
                             "event_rep": (rep.get("kpis") or {}).get("event_rep"), "week": week,
                             **{k: (v[week] if week < len(v) else None) for k, v in ts.items()}})
        return pd.DataFrame(rows)

    def dispatch(self, project_id, scenario_id, policy_version_id, *, acknowledge_warnings=False,
                 force_rerun=False, idempotency_key=None):
        """POST a run. Returns the run row (a reused one on 409, if you allow reuse)."""
        body = {"scenario_id": scenario_id, "policy_version_id": policy_version_id}
        if acknowledge_warnings:
            body["acknowledge_warnings"] = True
        if force_rerun:
            body["force_rerun"] = True
        # The key is derived from WHAT you ask for: re-running this cell replays the
        # same run, and changing any input dispatches a new one.
        key = idempotency_key or "nb-" + hashlib.sha256(
            json.dumps({"project": project_id, **body}, sort_keys=True).encode()).hexdigest()[:40]
        try:
            submitted = self.post(f"/projects/{project_id}/runs", body, idempotency_key=key)
        except SuReSuiteError as e:
            if e.code == "reuse_available":
                cand = (e.details or {}).get("reuse_candidate") or {}
                print(f"identical completed run already exists — reusing {cand.get('run_id')} "
                      f"(pass force_rerun=True to compute it again)")
                return self.run(cand["run_id"])
            if e.code == "validation_failed":
                d = e.details or {}
                print("the pre-run data gate stopped this run:")
                for f in d.get("findings") or []:
                    print(f"  - [{f.get('severity')}] {f.get('message')}")
                if d.get("validation") == "ack_required":
                    print("these are warnings: if they are acceptable, set ACKNOWLEDGE_WARNINGS = True in the CONFIG cell "
                          "(or pass acknowledge_warnings=True) and run this cell again")
            raise
        print(f"run {submitted['run_id']} dispatched ({submitted['status']})")
        return self.run(submitted["run_id"])

    def wait(self, run_id, timeout_seconds=1800):
        """Poll a run until it is done, failed or cancelled (backoff 3 s → 30 s)."""
        started, delay = time.time(), 3.0
        while True:
            run = self.run(run_id)
            if run["status"] in TERMINAL:
                print(f"\n→ {run['status']} after {time.time() - started:.0f}s"
                      f" — {run.get('rep_count_done') or 0} replications")
                if run["status"] == "failed":
                    print("error:", run.get("error_message"))
                return run
            done, target = run.get("rep_count_done") or 0, run.get("rep_count_target") or "?"
            print(f"  {run['status']} — replication {done}/{target}   ", end="\r")
            if time.time() - started > timeout_seconds:
                raise TimeoutError(f"run {run_id} still {run['status']} after {timeout_seconds}s")
            if self.live:
                time.sleep(delay)
                delay = min(delay * 1.5, 30.0)

    def run_and_wait(self, project_id, scenario_id, policy_version_id, **kw):
        run = self.dispatch(project_id, scenario_id, policy_version_id, **kw)
        return run if run["status"] in TERMINAL else self.wait(run["id"])


# ── results as tables ───────────────────────────────────────────────────────
def _fmt(kind, v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return "—"
    if kind == "pct2":
        return f"{v * 100:.2f}%"
    if kind == "pct1":
        return f"{v * 100:.1f}%"
    if kind == "money":
        return f"{'-' if v < 0 else ''}{_MONEY['symbol']}{abs(v):,.0f}"
    if kind == "units":
        return f"{v:,.0f}"
    if kind.startswith("fixed"):
        return f"{v:.{int(kind[5:] or 2)}f}"
    return f"{v:,.4g}"


def kpi_label(key):
    return KPI_DISPLAY.get(key, (key, "num", None))[0]


def fmt_kpi(key, value):
    return _fmt(KPI_DISPLAY.get(key, (key, "num", None))[1], value)


def _numeric(d):
    return {k: v for k, v in (d or {}).items()
            if not k.startswith("_") and isinstance(v, (int, float)) and not isinstance(v, bool)
            and not math.isnan(v)}


def _order(keys):
    rank = {k: i for i, k in enumerate(KPI_DISPLAY)}
    return sorted(keys, key=lambda k: (rank.get(k, 10_000), k))


def kpi_table(run, kpis=None):
    """A run's aggregate KPIs: mean ± 95 % CI half-width, labelled and formatted the way
    the Simulation Lab shows them. Measures the engine did not produce are left out."""
    agg, ci = _numeric(run.get("aggregate_kpis")), run.get("ci_half_widths") or {}
    keys = [k for k in (kpis or _order(agg)) if k in agg]
    return pd.DataFrame([{"kpi": k, "measure": kpi_label(k), "mean": fmt_kpi(k, agg[k]),
                          "± 95% CI": fmt_kpi(k, ci.get(k)) if ci.get(k) is not None else "—",
                          "value": agg[k]} for k in keys],
                        columns=["kpi", "measure", "mean", "± 95% CI", "value"]).set_index("kpi")


def replications_frame(replications):
    """One row per replication: rep_index, model_rep, event_rep and every KPI."""
    rows = [{"rep_index": r["rep_index"], **(r.get("kpis") or {})} for r in replications]
    return pd.DataFrame(rows, columns=None if rows else ["rep_index"]).set_index("rep_index").sort_index()


def capacity_binding(run):
    """Which products / suppliers capacity held back, if any (from the run's _meta)."""
    return ((run.get("aggregate_kpis") or {}).get("_meta") or {}).get("capacity_binding")


def assert_disruption_applied(replications):
    """A disruption that reached the engine leaves recovery measures on every replication.
    None of them means every event was SKIPPED (a target the engine cannot disrupt) and
    the run is really a baseline — say so instead of reporting its KPIs as a stress test."""
    if not any("recovery_measurable" in (r.get("kpis") or {}) for r in replications):
        raise AssertionError(
            "no disruption reached the engine: every event was skipped. A target must be one of "
            "this project's supplier ids or the plant (\"plant\") — material, customer and lane "
            "targets are not supported by the engine yet.")
    print("disruption applied — recovery measures are present on the replications")


def paired_compare(reps_a, reps_b, kpis=None, labels=("A", "B")):
    """B − A per KPI, paired replication by replication on the common-random-numbers
    cell (model_rep, event_rep) — the Simulation Lab's Compare rule. A direction
    ("better" / "worse") is drawn only when the 95 % interval of the paired
    difference excludes zero; otherwise the honest answer is "no clear difference"."""

    def key(r):
        k = r.get("kpis") or {}
        return (f"m{int(k['model_rep'])}:e{int(k.get('event_rep') or 0)}"
                if isinstance(k.get("model_rep"), (int, float)) else f"i{r['rep_index']}")

    a_by = {key(r): r for r in reps_a if r.get("status", "done") == "done"}
    keys = kpis or _order(set().union(*[_numeric(r.get("kpis")).keys() for r in reps_a])
                          - {"model_rep", "event_rep", "recovery_measurable", "ttr_censored", "tts_censored"})
    rows = []
    for kpi in keys:
        diffs, va_all, vb_all = [], [], []
        for r in reps_b:
            a = a_by.get(key(r))
            va = _numeric((a or {}).get("kpis")).get(kpi)
            vb = _numeric(r.get("kpis")).get(kpi)
            if a is not None and va is not None and vb is not None and r.get("status", "done") == "done":
                diffs.append(vb - va)
                va_all.append(va)
                vb_all.append(vb)
        n = len(diffs)
        if n < 2:
            continue
        m = sum(diffs) / n
        sd = math.sqrt(sum((d - m) ** 2 for d in diffs) / (n - 1))
        hw = (T95.get(n - 1, 1.96) * sd) / math.sqrt(n)
        separated = abs(m) > hw
        direction = KPI_DISPLAY.get(kpi, (kpi, "num", None))[2]
        if not separated:
            verdict = "no clear difference"
        elif direction is None:
            verdict = "higher" if m > 0 else "lower"
        else:
            verdict = f"{labels[1]} better" if (m > 0) == direction else f"{labels[1]} worse"
        rows.append({"kpi": kpi, "measure": kpi_label(kpi),
                     labels[0]: fmt_kpi(kpi, sum(va_all) / n), labels[1]: fmt_kpi(kpi, sum(vb_all) / n),
                     f"{labels[1]} − {labels[0]}": f"{'+' if m >= 0 else '-'}{fmt_kpi(kpi, abs(m))} ± {fmt_kpi(kpi, hw)}",
                     "pairs": n, "verdict": verdict, "delta": m, "half_width": hw})
    return pd.DataFrame(rows).set_index("kpi") if rows else pd.DataFrame()


@contextlib.contextmanager
def editing_policies(api, project_id):
    """Edit the project's live policy defaults inside this block; they are put back
    exactly as they were when the block ends — even if a run fails. (Per-node
    overrides are not touched: the API has no way to delete one.)"""
    original = api.policy_defaults(project_id)
    try:
        yield original
    finally:
        api.put(f"/projects/{project_id}/policies", {"defaults": original})
        print("policies restored to their original values")


# ── picking what to work on ─────────────────────────────────────────────────
FRAME_FIELDS = ("horizon_days", "warmup_days", "replications", "seed", "crn", "primary_kpi")


def pick_project(api, project_id=""):
    """The configured project, or the first one this key can see."""
    projects = api.projects()
    if not projects:
        raise Stop("this key can see no projects — check its organization and project restriction on /developer")
    if project_id and project_id not in {p["id"] for p in projects}:
        raise Stop(f"PROJECT_ID {project_id} is not a project this key can see — set PROJECT_ID = \"\" in the "
                   "CONFIG cell to use the first one, or download the notebook again from /developer")
    project = api.project(project_id) if project_id else projects[0]
    print(f"project: {project['name']}  ({project['id']})")
    return project


def baseline_scenario(api, project_id, scenario_id=""):
    """The configured scenario, else the newest one WITHOUT disruptions, else a new
    52-week baseline (the Simulation Lab's defaults: 364 days, 10 replications,
    seed 42, common random numbers on)."""
    scenarios = api.scenarios(project_id)
    if scenario_id:
        sc = next((s for s in scenarios if s["id"] == scenario_id), None)
        if sc is None:
            raise Stop(f"SCENARIO_ID {scenario_id} is not a scenario of project {project_id} — set "
                       "SCENARIO_ID = \"\" in the CONFIG cell to pick or create a baseline")
    else:
        sc = next((s for s in scenarios if not s.get("disruption_schedule")), None)
        if sc is None:
            created = api.create_scenario(project_id, "Notebook baseline", description="52 weeks, no disruption",
                                          horizon_days=364, replications=10, seed=42, crn=True)
            sc = next(s for s in api.scenarios(project_id) if s["id"] == created["id"])
            print("created a baseline scenario")
    if sc.get("disruption_schedule"):
        print("note: this scenario already has disruptions — it is not a clean baseline")
    if (sc.get("horizon_days") or 0) < 364:
        print(f"note: horizon_days={sc.get('horizon_days')} runs as 52 weeks — the engine's minimum horizon")
    print(f"scenario: {sc['name']}  ({sc['id']}) — {sc['horizon_days']} days, {sc['replications']} replications, "
          f"seed {sc['seed']}, CRN {'on' if sc.get('crn') else 'off'}")
    return sc


def baseline_policy_version(api, project_id, version_id="", label="notebook baseline"):
    """The configured policy version, else a snapshot of the current policies (an
    unchanged configuration returns the version that already holds it), else — for a
    key without `write:policies` — the newest saved version, and it says so."""
    versions = api.policy_versions(project_id)
    if version_id:
        v = next((v for v in versions if v["id"] == version_id), None)
        if v is None:
            raise Stop(f"POLICY_VERSION_ID {version_id} is not a policy version of project {project_id} — set "
                       "POLICY_VERSION_ID = \"\" in the CONFIG cell to use the current policies")
        return v
    try:
        return api.snapshot_policies(project_id, label)
    except SuReSuiteError as e:
        if e.code != "missing_scope":
            raise
    if not versions:
        raise Stop("this project has no saved policy version and this key cannot save one (no write:policies) "
                   "— save one in the app (/policies → Save version) and run again")
    newest = max(versions, key=lambda v: v.get("created_at") or "")
    print(f"note: this key cannot snapshot policies (no write:policies), so the newest saved version is used: "
          f"{newest.get('label') or 'unlabelled'} — it may differ from the policies as they are now")
    return newest


def require_done(run):
    """Stop here, with the reason, unless the run finished."""
    if run["status"] != "done":
        raise Stop(f"run {run['id']} ended {run['status']}"
                   + (f": {run.get('error_message')}" if run.get("error_message") else "")
                   + " — nothing below has results to read")
    return run


def stress_scenario(api, project_id, base, events, name):
    """`base`'s frame (horizon, replications, seed, CRN) plus a disruption schedule.
    Reuses an existing scenario with the same name and schedule, so running a cell
    again does not pile up copies in the Simulation Lab."""
    norm = lambda sched: _canon([{k: e[k] for k in ("target", "start_day", "duration_days", "magnitude_pct")}
                                 for e in sched or []])
    for sc in api.scenarios(project_id):
        if sc["name"] == name and norm(sc.get("disruption_schedule")) == norm(events):
            return sc
    created = api.create_scenario(project_id, name, description="created by a SuReSuite notebook",
                                  disruption_schedule=events,
                                  **{k: base[k] for k in FRAME_FIELDS if base.get(k) is not None})
    return next(s for s in api.scenarios(project_id) if s["id"] == created["id"])


def outage(target, start_day=140, duration_days=42, magnitude_pct=100):
    """One disruption event. target: a supplier id of the project, or "plant".
    magnitude_pct 100 = the target ships nothing for the window (its deliveries are
    deferred, not lost); below 100 = its capacity is cut BY that share, which bites
    only when the remaining capacity is below what the plan needs."""
    return {"target": target, "target_type": "node", "start_day": start_day,
            "duration_days": duration_days, "magnitude_pct": magnitude_pct}


# ── the run-results workbook (the app's "verifiable export") ────────────────
def export_run_workbook(api, run, scenario, path, policy_version_label=None):
    """Write the sheets the app's run-results workbook carries — run_meta,
    aggregate_kpis, replication_kpis, series_<key> (weeks × replications) and
    reproducibility — so a figure that leaves the notebook says what produced it."""
    reps = [r for r in api.replications(run["id"]) if r.get("status", "done") == "done"]
    agg, ci = run.get("aggregate_kpis") or {}, run.get("ci_half_widths") or {}
    rng = agg.get("_range") or {}
    validation = api.validation(run["id"])
    meta = [
        ("run_id", run["id"]), ("status", run["status"]), ("created_at", run.get("created_at")),
        ("ended_at", run.get("ended_at")), ("code_version (engine)", run.get("code_version")),
        ("policy_version_id", run.get("policy_version_id")), ("policy_version_label", policy_version_label),
        ("policy_hash (SHA-256)", run.get("policy_hash")), ("dataset_version_id", run.get("dataset_version_id")),
        ("graph_hash (SHA-256)", run.get("graph_hash")), ("scenario_hash (baseline fingerprint)", run.get("scenario_hash")),
        ("model_validation_id", run.get("model_validation_id")), ("scenario_id", scenario["id"]),
        ("scenario_name", scenario["name"]), ("root seed", scenario.get("seed")),
        ("replications (target)", run.get("rep_count_target")), ("replications (done)", run.get("rep_count_done")),
        ("CRN (common random numbers)", scenario.get("crn")), ("horizon_days", scenario.get("horizon_days")),
        ("warmup_detected_at (weeks, engine)", run.get("warmup_detected_at")),
        ("disruption_schedule (full JSON; empty = baseline)", json.dumps(scenario.get("disruption_schedule") or [])),
        ("credibility", validation.get("status")), ("source", "SuReSuite public API /v1" if api.live else "offline demo"),
        ("generated", pd.Timestamp.now(tz="UTC").isoformat()),
    ]
    aggregate = pd.DataFrame([{"kpi": k, "mean": agg[k], "ci_halfwidth": ci.get(k),
                               "min": (rng.get(k) or {}).get("min"), "max": (rng.get(k) or {}).get("max")}
                              for k in sorted(_numeric(agg))])
    replication = pd.DataFrame([{"rep_index": r["rep_index"], "seed_used (root seed)": r.get("seed_used"),
                                 **(r.get("kpis") or {})} for r in reps])
    series = api.series(run["id"])
    label = lambda row: f"rep {int(row.rep_index)}" + (f" · world {int(row.model_rep)}" if pd.notna(row.model_rep) else "")
    series["replication"] = series.apply(label, axis=1)
    repro = [
        ("dataset", f"dataset version {run.get('dataset_version_id')} · graph_hash {run.get('graph_hash')}"),
        ("policy", f"policy version {run.get('policy_version_id')} · policy_hash {run.get('policy_hash')}"),
        ("scenario", f"scenario {scenario['id']} · scenario_hash {run.get('scenario_hash')} · seed {scenario.get('seed')}"
                     f" · disruptions {json.dumps(scenario.get('disruption_schedule') or [])}"),
        ("engine", run.get("code_version")),
        ("known limit", "assembled by a notebook from the API's run row; network-analysis figures and "
                        "mapping warnings are not part of this record (the API does not return them)"),
    ]
    with pd.ExcelWriter(path) as xl:
        pd.DataFrame(meta, columns=["field", "value"]).to_excel(xl, sheet_name="run_meta", index=False)
        aggregate.to_excel(xl, sheet_name="aggregate_kpis", index=False)
        replication.to_excel(xl, sheet_name="replication_kpis", index=False)
        for col in [c for c in series.columns if c not in ("rep_index", "model_rep", "event_rep", "week", "replication")]:
            series.pivot_table(index="week", columns="replication", values=col, sort=False).to_excel(
                xl, sheet_name=f"series_{col}"[:31])
        pd.DataFrame(repro, columns=["binding", "value"]).to_excel(xl, sheet_name="reproducibility", index=False)
    print(f"wrote {path}: run_meta, aggregate_kpis, replication_kpis, {series.shape[1] - 5} series sheets, reproducibility")
    return path


# ── charts ──────────────────────────────────────────────────────────────────
def _need_plt():
    if plt is None:
        raise ImportError("charts need matplotlib: pip install 'suresuite[plots]'")


def plot_kpis(runs, kpis, title=None):
    """One small panel per KPI (their scales differ), one bar per run, with the
    run's 95 % CI half-width as the error bar. `runs` is {label: run_row}."""
    _need_plt()
    kpis = [k for k in kpis if all(k in _numeric(r.get("aggregate_kpis")) for r in runs.values())]
    if not kpis:
        print("none of those KPIs is present on every run")
        return
    fig, axes = plt.subplots(1, len(kpis), figsize=(2.9 * len(kpis), 3.3), squeeze=False)
    for ax, kpi in zip(axes[0], kpis):
        labels = list(runs)
        vals = [_numeric(runs[l].get("aggregate_kpis"))[kpi] for l in labels]
        errs = [(runs[l].get("ci_half_widths") or {}).get(kpi) or 0 for l in labels]
        ax.bar(range(len(labels)), vals, yerr=errs, capsize=3, width=0.65,
               color=[PALETTE[i % len(PALETTE)] for i in range(len(labels))])
        ax.set_xticks(range(len(labels)), labels, rotation=30, ha="right", fontsize=8)
        ax.set_title(kpi_label(kpi), fontsize=9)
        ax.spines[["top", "right"]].set_visible(False)
    if title:
        fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


def plot_series(series_by_label, column, title=None, window=None, warmup_week=None):
    """Mean weekly trajectory per run with a band from the lowest to the highest
    replication. `series_by_label` is {label: api.series(run_id)}; `window` is a
    (start_week, end_week) disruption window to shade."""
    _need_plt()
    if not any(df is not None for df in series_by_label.values()):
        print("no weekly series to plot")
        return
    fig, ax = plt.subplots(figsize=(8, 3.4))
    for i, (label, df) in enumerate(series_by_label.items()):
        if df is None or column not in df:
            continue
        g = df.groupby("week")[column]
        color = PALETTE[i % len(PALETTE)]
        ax.plot(g.mean().index, g.mean().values, color=color, linewidth=2, label=label)
        ax.fill_between(g.min().index, g.min().values, g.max().values, color=color, alpha=0.12, linewidth=0)
    if window:
        ax.axvspan(window[0], window[1], color="#5f6b7a", alpha=0.10, label="disruption window")
    if warmup_week:
        ax.axvline(warmup_week, color="#5f6b7a", linestyle=":", linewidth=1, label="warm-up ends")
    ax.set_xlabel("week")
    ax.set_ylabel(column)
    ax.set_title(title or column, fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(fontsize=8, frameon=False, loc="best")
    plt.tight_layout()
    plt.show()


def effective_window(event, run):
    """The weeks an event actually covered: the engine works in weeks, starts no
    earlier than week 1, and moves an event that would start inside the warm-up to
    the warm-up's end."""
    start = max(1, round(event["start_day"] / 7))
    start = max(start, int(run.get("warmup_detected_at") or 0))
    return start, start + max(1, min(52, round(event["duration_days"] / 7)))


# ── the offline demo: the same API, answered from recorded engine output ─────
RUN_FIELDS = ("id", "scenario_id", "project_id", "status", "started_at", "ended_at", "created_at",
              "updated_at", "rep_count_target", "rep_count_done", "aggregate_kpis", "ci_half_widths",
              "warmup_detected_at", "policy_version_id", "policy_hash", "dataset_version_id",
              "graph_hash", "scenario_hash", "model_validation_id", "gate_skipped", "error_message",
              "code_version")
SCENARIO_FIELDS = {"name": None, "description": "", "horizon_days": 90, "warmup_days": 14,
                   "replications": 10, "seed": 42, "crn": True, "disruption_schedule": [],
                   "recovery_overrides": {}, "primary_kpi": "fill_rate"}
RUN_BODY_FIELDS = ("scenario_id", "policy_version_id", "acknowledge_warnings", "force_rerun")


def _canon(v):
    return json.dumps(v, sort_keys=True, separators=(",", ":"))


def _sha(v):
    return hashlib.sha256(_canon(v).encode()).hexdigest()


def _demo_key(policies, schedule, frame):
    """MUST match demo_key() in notebooks/tools/record_demo.py (pinned by the tests)."""
    pol = {f: v for f, v in sorted(policies.items()) if v}
    sched = sorted(({"target": str(e["target"]), "start_day": int(e["start_day"]),
                     "duration_days": int(e["duration_days"]),
                     "magnitude_pct": float(e.get("magnitude_pct", 100))} for e in schedule), key=_canon)
    fr = {k: frame[k] for k in ("horizon_days", "replications", "seed", "crn")}
    return _sha({"policies": pol, "schedule": sched, "frame": fr})


class DemoTransport:
    """Answers the /v1 routes the notebooks use from recorded engine output, keeping
    policies, scenarios and runs in memory for this session. Responses have the live
    API's shapes; a request it has no recording for is refused, never invented."""

    TS = "2026-10-01T00:00:00+00:00"

    def __init__(self, data):
        self.d = data
        self.pid = data["project"]["id"]
        self.recordings = {r["key"]: r for r in data["recordings"]}
        self.supplier_ids = {s["supplier_id"] for s in data["dataset"]["suppliers"]}
        self.defaults = {f: {} for f in FAMILIES}
        self.overrides = []
        self.datasets = [{"id": self._id("dataset", 1), "label": "initial upload", "graph_hash": data["graph_hash"],
                          "author_email": "demo@example.com", "created_at": self.TS, "version_no": 1, "tuple": None}]
        self.versions, self.scenarios, self.runs, self.idem = [], [], {}, {}
        self._new_scenario({"name": "Baseline (demo)", "description": "52 weeks, no disruption",
                            "horizon_days": 364, "replications": 10})
        self._n = 0

    def _id(self, kind, n):
        return str(uuid.uuid5(uuid.NAMESPACE_URL, f"suresuite-demo/{kind}/{n}"))

    def _err(self, status, code, message, details=None):
        return status, {"X-Request-Id": "demo"}, {"error": {"code": code, "message": message,
                                                            **({"details": details} if details else {})}}

    def _ok(self, body, status=200):
        return status, {"X-Request-Id": "demo"}, body

    def _new_scenario(self, body):
        unknown = set(body) - set(SCENARIO_FIELDS)
        if unknown:
            raise ValueError(f"unknown scenario field(s): {sorted(unknown)}")
        sc = {**copy.deepcopy(SCENARIO_FIELDS), **copy.deepcopy(body),
              "id": self._id("scenario", len(self.scenarios) + 1), "created_at": self.TS, "updated_at": self.TS}
        self.scenarios.insert(0, sc)
        return sc

    def fetch(self, url):
        rec = self.recordings[self.runs[url.split("demo://series/")[1]]["_key"]]
        return rec["series"]  # already a column table; read by series() below

    def request(self, method, path, params=None, body=None, headers=None):
        params, body, headers = params or {}, body or {}, headers or {}
        p = path.split("?")[0]
        m = re.fullmatch(r"/projects/([^/]+)(/.*)?", p)
        if p == "/projects" and method == "GET":
            return self._ok({"data": [self.d["project"]], "next_cursor": None})
        if m:
            if m.group(1) != self.pid:
                return self._err(404, "project_not_found", "project not found")
            return self._project(method, m.group(2) or "", body, headers)
        if p == "/key" and method == "GET":
            return self._ok({"key_prefix": "demo", "env": "test", "principal": "personal",
                             "scopes": [f"{a}:{b}" for b in ("data", "policies", "runs") for a in ("read", "write")],
                             "project_ids": None})
        if p == "/engine" and method == "GET":
            return self._err(403, "demo_no_engine",
                             "the offline demo cannot hand out the engine — it comes with an API key "
                             "(create one on the app's /developer page, then suresuite.install_engine())")
        m = re.fullmatch(r"/runs/([^/:]+)(/replications|/validation|:cancel|:add-reps)?", p)
        if m:
            run = self.runs.get(m.group(1))
            if run is None:
                return self._err(404, "run_not_found", "run not found")
            return self._run(method, run, m.group(2) or "", params, body)
        return self._err(404, "route_not_found", f"no route for {method} {p}")

    def _project(self, method, sub, body, headers):
        if sub == "" and method == "GET":
            return self._ok(self.d["project"])
        if sub == "/dataset-versions" and method == "GET":
            return self._ok({"data": self.datasets})
        if sub == "/datasets:freeze" and method == "POST":
            dv = self.datasets[0]  # nothing changed since: the existing version comes back
            return self._ok({k: dv[k] for k in ("id", "label", "graph_hash", "created_at")}, 201)
        if sub == "/policy-catalog" and method == "GET":
            return self._ok(self.d["catalog"])
        if sub == "/policies" and method == "GET":
            return self._ok({"defaults": {"project_id": self.pid, **copy.deepcopy(self.defaults),
                                          "fulfillment_strategy": None, "active_preset": None,
                                          "created_at": self.TS, "updated_at": self.TS},
                             "overrides": copy.deepcopy(self.overrides)})
        if sub == "/policies" and method == "PUT":
            bad = set(body) - {"defaults", "fulfillment_strategy", "overrides"}
            bad |= set((body.get("defaults") or {})) - set(FAMILIES)
            if bad:
                return self._err(400, "invalid_request", f"unrecognized key(s): {sorted(bad)}")
            for fam, val in (body.get("defaults") or {}).items():
                self.defaults[fam] = copy.deepcopy(val)
            for o in body.get("overrides") or []:
                self.overrides = [x for x in self.overrides if (x["scope"], x["target_key"], x["family"])
                                  != (o["scope"], o["target_key"], o["family"])] + [{**o, "updated_at": self.TS}]
            return self._ok({"ok": True})
        if sub == "/policy-versions" and method == "POST":
            h = _sha({f: v for f, v in self.defaults.items() if v})
            for v in self.versions:  # content-addressed: the oldest holder of this config comes back
                if v["policy_hash"] == h:
                    return self._ok({k: v[k] for k in ("id", "label", "policy_hash", "created_at")}, 201)
            v = {"id": self._id("policy-version", len(self.versions) + 1), "label": body.get("label"),
                 "notes": None, "author_email": "demo@example.com", "author_name": "Demo",
                 "parent_version_id": None, "policy_hash": h, "created_at": self.TS, "run_count": 0,
                 "card_count": 0, "version_no": len(self.versions) + 1, "_policies": copy.deepcopy(self.defaults)}
            self.versions.append(v)
            return self._ok({k: v[k] for k in ("id", "label", "policy_hash", "created_at")}, 201)
        m = re.fullmatch(r"/dataset-versions/([^/]+)", sub)
        if m and method == "GET":
            dv = self.datasets[0] if m.group(1) in ("latest", self.datasets[0]["id"]) else None
            if dv is None:
                return self._err(404, "dataset_version_not_found", "no such dataset version in this project")
            snap = {"schema_version": 3, "inputs": {**copy.deepcopy(self.d["dataset"]), "customers": [],
                                                    "bom_multi_level": []}, "network": {}}
            return self._ok({**{k: dv[k] for k in ("id", "label", "version_no", "graph_hash", "author_email",
                                                   "created_at")},
                             "hash_inputs": None, "hash_network": None, "schema_version": 3, "snapshot": snap})
        m = re.fullmatch(r"/policy-versions/([^/]+)", sub)
        if m and method == "GET":
            vid = m.group(1)
            v = (self.versions[-1] if self.versions else None) if vid == "latest" else \
                next((x for x in self.versions if x["id"] == vid), None)
            if v is None:
                return self._err(404, "policy_version_not_found", "no such policy version in this project")
            return self._ok({**{k: x for k, x in v.items() if not k.startswith("_")},
                             "snapshot": {"schema_version": 2, "defaults": copy.deepcopy(v["_policies"]),
                                          "fulfillment_strategy": None, "overrides": []}})
        if sub == "/policy-versions" and method == "GET":
            return self._ok({"data": [{k: v for k, v in x.items() if not k.startswith("_")}
                                      for x in reversed(self.versions)]})
        if sub == "/scenarios" and method == "GET":
            return self._ok({"data": copy.deepcopy(self.scenarios), "next_cursor": None})
        if sub == "/scenarios" and method == "POST":
            try:
                sc = self._new_scenario(body)
            except ValueError as e:
                return self._err(400, "invalid_request", str(e))
            return self._ok({k: sc[k] for k in ("id", "name", "horizon_days", "replications", "seed", "created_at")}, 201)
        if sub == "/runs" and method == "POST":
            return self._dispatch(body, headers.get("Idempotency-Key"))
        return self._err(404, "route_not_found", f"no route for {method} /projects/…{sub}")

    def _dispatch(self, body, idem):
        bad = set(body) - set(RUN_BODY_FIELDS)
        if bad:
            return self._err(400, "invalid_request", f"unrecognized key(s): {sorted(bad)}")
        if idem and idem in self.idem:
            run = self.runs[self.idem[idem]]
            return 202, {"Idempotency-Replayed": "true"}, {"run_id": run["id"], "status": run["status"],
                                                          "policy_hash": run["policy_hash"], "graph_hash": run["graph_hash"]}
        sc = next((s for s in self.scenarios if s["id"] == body.get("scenario_id")), None)
        ver = next((v for v in self.versions if v["id"] == body.get("policy_version_id")), None)
        if sc is None or ver is None:
            return self._err(500, "dispatch_failed", "scenario or policy version does not belong to this project")
        findings = []
        for ev in sc["disruption_schedule"][:5]:
            raw = str(ev.get("target", ""))
            tid = raw.rsplit(":", 1)[-1]
            if tid in self.supplier_ids or raw.lower().startswith("plant:") or tid.lower() == "plant":
                continue
            kind = raw.split(":", 1)[0].lower() if ":" in raw else ""
            why = (f"The engine cannot disrupt {kind} targets yet — this event is skipped."
                   if kind in ("material", "edge", "customer", "lane")
                   else f"No supplier or plant named \"{tid}\" in this project's data — this event is skipped.")
            findings.append({"severity": "warn", "field": "scenarios.disruption_schedule", "policy": "engine",
                             "rows": [raw], "message": f"Disruption on \"{raw}\": {why}"})
        if findings and not body.get("acknowledge_warnings"):
            return self._err(422, "validation_failed", "the pre-run gate needs acknowledgement",
                             {"validation": "ack_required", "ack_required": True, "findings": findings})
        live_schedule = [ev for ev in sc["disruption_schedule"][:5]
                         if not any(f["rows"] == [str(ev.get("target", ""))] for f in findings)]
        key = _demo_key(ver["_policies"], live_schedule, sc)
        rec = self.recordings.get(key)
        if rec is None:
            labels = ", ".join(r["label"] for r in self.d["recordings"])
            return self._err(400, "demo_no_recording",
                             "the offline demo has no recording for this policy + scenario combination "
                             f"(recorded: {labels}). Use the notebook's suggested settings, or go live.")
        if not body.get("force_rerun"):
            for prev in self.runs.values():
                if prev["_key"] == key and prev["policy_version_id"] == ver["id"] and prev["status"] == "done":
                    return self._err(409, "reuse_available", "an identical completed run exists",
                                     {"reuse_candidate": {"run_id": prev["id"], "ended_at": prev["ended_at"],
                                                          "created_at": prev["created_at"],
                                                          "code_version": prev["code_version"],
                                                          "rep_count_done": prev["rep_count_done"]}})
        self._n += 1
        run = {f: None for f in RUN_FIELDS}
        run.update(id=self._id("run", self._n), scenario_id=sc["id"], project_id=self.pid, status="queued",
                   created_at=self.TS, updated_at=self.TS, rep_count_target=sc["replications"], rep_count_done=0,
                   policy_version_id=ver["id"], policy_hash=ver["policy_hash"],
                   dataset_version_id=self.datasets[0]["id"], graph_hash=self.d["graph_hash"],
                   scenario_hash=_sha({k: sc[k] for k in ("horizon_days", "warmup_days", "replications", "seed", "crn")}),
                   gate_skipped=False, _key=key, _polls=0)
        self.runs[run["id"]] = run
        if idem:
            self.idem[idem] = run["id"]
        return 202, {"X-Request-Id": "demo"}, {"run_id": run["id"], "status": "queued", "policy_hash": run["policy_hash"],
                                                "graph_hash": run["graph_hash"], "gate_skipped": False}

    def _advance(self, run):
        if run["status"] in TERMINAL:
            return
        run["_polls"] += 1
        rec = self.recordings[run["_key"]]
        if run["_polls"] == 1:
            run.update(status="running", started_at=self.TS, rep_count_done=run["rep_count_target"] // 2)
        else:
            res = rec["run"]
            run.update(status="done", ended_at=self.TS, rep_count_done=res["rep_count_done"],
                       aggregate_kpis=copy.deepcopy(res["aggregate_kpis"]),
                       ci_half_widths=copy.deepcopy(res["ci_half_widths"]),
                       warmup_detected_at=res.get("warmup_detected_at"), code_version=res["code_version"])

    def _run(self, method, run, sub, params, body):
        if sub == "" and method == "GET":
            self._advance(run)
            return self._ok({k: copy.deepcopy(run[k]) for k in RUN_FIELDS})
        if sub == "/replications" and method == "GET":
            reps = self.recordings[run["_key"]]["replications"] if run["status"] == "done" else []
            after = int(params.get("cursor")) if params.get("cursor") not in (None, "") else -1
            limit = max(1, min(100, int(params.get("limit") or 20)))
            rows = [copy.deepcopy(r) for r in reps if r["rep_index"] > after]
            page = rows[:limit]
            body = {"data": page, "next_cursor": str(page[-1]["rep_index"]) if len(rows) > limit else None}
            if params.get("include") == "time_series":
                for r in page:
                    r["time_series"] = {}
                body["series"] = {"location": "object", "format": "parquet (zstd), long form",
                                  "url": f"demo://series/{run['id']}", "expires_in_seconds": 600}
            return self._ok(body)
        if sub == "/validation" and method == "GET":
            return self._ok({"status": "unvalidated", "model_validation": None, "gate_skipped": False})
        if sub == ":cancel" and method == "POST":
            if run["status"] not in TERMINAL:
                run.update(status="cancelled", ended_at=self.TS)
            # Like the live API, the answer says "cancelled" even for a run that had
            # already finished — read the run back to see what really happened.
            return self._ok({"run_id": run["id"], "status": "cancelled"}, 202)
        if sub == ":add-reps" and method == "POST":
            return self._ok({"run_id": run["id"], "added": (body or {}).get("n")}, 202)
        return self._err(404, "route_not_found", f"no route for {method} /runs/…{sub}")


def _demo(demo_data):
    if isinstance(demo_data, str):  # the notebook embeds it compressed (raw deflate + base64)
        demo_data = json.loads(zlib.decompress(base64.b64decode(demo_data), -15))
    return SuReSuite(DemoTransport(demo_data), live=False)


def connect(base_url, api_key=None, demo_data=None, needs=(), mode="auto"):
    """A live client when you have a key, otherwise the offline demo.

    `needs` are the scopes the caller will use. A live key is checked against them
    BEFORE anything runs: one the platform would refuse partway through stops here
    and says what is missing — or, in `mode="auto"` with demo data at hand, says so
    and continues as the offline demo, so a notebook never breaks halfway."""
    if not api_key:
        return _demo(demo_data)
    api = SuReSuite(HttpTransport(base_url, api_key), live=True)
    if not needs:
        return api
    info = api.key()
    if info is None:  # a gateway without GET /key: nothing to check against
        return api
    missing = [s for s in needs if s not in (info.get("scopes") or [])]
    if not missing:
        return api
    why = (f"this key ({info.get('principal')} key sk_{info.get('env')}_{info.get('key_prefix')}…) cannot use "
           f"{', '.join(missing)}, which this notebook needs. Create a key on the app's /developer page with "
           f"{', '.join(needs)}, then set SURESUITE_API_KEY to it and run all cells again.")
    if info.get("principal") == "personal" and not any(s.startswith("write:") for s in info.get("scopes") or []):
        why += (" A personal key is read-only unless its owner is an organization admin or a modeler — "
                "ask one of them for the role or for a key.")
    if mode == "live" or demo_data is None:
        raise Stop(why)
    print(f"{why}\n\nUntil then this notebook runs in DEMO mode on the example project, so every cell "
          "still works.")
    return _demo(demo_data)


# GENERATED by scripts/notebooks/build-notebooks.mjs from src/lib/sim/kpiDisplay.ts — do not edit.
# The Simulation Lab's KPI vocabulary: key → (label, format, higher_is_better).
# higher_is_better is None where "better" is not defined (inventory, utilization).
KPI_DISPLAY = {
    "fill_rate": ("Fill rate (α)", "pct2", True),
    "revenue": ("Revenue", "money", True),
    "demand_value": ("Demand (value)", "money", None),
    "produced_value": ("Produced (value)", "money", None),
    "lost_sales_value": ("Lost sales (value)", "money", False),
    "lost_units": ("Lost units", "units", False),
    "lost_inbound_units": ("Lost inbound units", "units", False),
    "max_backlog": ("Peak backlog (units)", "units", False),
    "avg_on_hand_value": ("Average material inventory (value)", "money", None),
    "avg_fg_value": ("Average finished-goods inventory (value)", "money", None),
    "avg_on_hand_units": ("Average material inventory (units)", "units", None),
    "avg_fg_units": ("Average finished-goods inventory (units)", "units", None),
    "capacity_utilization": ("Plant capacity utilization", "pct1", None),
    "supplier_capacity_utilization": ("Supplier capacity utilization", "pct1", None),
    "products_capacity_bound": ("Products capacity held back", "units", False),
    "suppliers_capacity_bound": ("Suppliers capacity held back", "units", False),
    "cost_of_resilience": ("Cost of resilience", "money", False),
    "cost_ss_holding": ("Cost · safety-stock holding", "money", False),
    "cost_backup_premium": ("Cost · backup supplier premium", "money", False),
    "cost_multi_sourcing_premium": ("Cost · multi-sourcing premium", "money", False),
    "cost_expediting": ("Cost · expediting freight", "money", False),
    "cost_overtime": ("Cost · overtime capacity", "money", False),
    "cost_lost_sales": ("Cost · lost sales", "money", False),
    "cost_allocation_labor": ("Cost · allocation labour", "money", False),
    "cost_fg_ss_holding": ("Cost · finished-goods safety stock", "money", False),
    "cost_backorder_penalty": ("Cost · backorder penalty", "money", False),
    "cost_monitoring": ("Cost · supplier monitoring", "money", False),
    "ttr_weeks": ("Time to recover (weeks)", "fixed1", False),
    "tts_weeks": ("Time to survive (weeks)", "fixed1", True),
    "pre_disruption_fill_rate": ("Fill rate before the disruption", "pct2", True),
    "fill_rate_beta": ("Fill rate (β)", "pct2", True),
    "otif": ("OTIF", "pct2", True),
    "lead_time_days": ("Lead time (days)", "fixed2", False),
    "lead_time_p95": ("Lead time p95", "fixed2", False),
    "cost": ("Cost", "money", False),
    "profit": ("Profit", "money", True),
    "utilization": ("Utilization (avg) · legacy", "pct1", True),
    "inventory_turns": ("Inventory turns", "fixed2", True),
    "backorder_days": ("Backorder days", "fixed1", False),
    "ttr_days": ("Time-to-recover (days) · legacy", "fixed1", False),
}

In [ ]:
# @title Local simulation (the suresuite package's local.py)
"""Simulate on your own machine — Phase 12 · WP 12.5.

Pull the frozen inputs a platform run reads, then run the SAME engine function
the platform's browser engine and the Fly worker use
(`sim_worker.local.run_from_snapshots`), on your own CPU:

    import suresuite as ss
    api = ss.connect()                                  # key from SURESUITE_API_KEY
    ss.install_engine(api)                              # once per environment
    data = ss.dataset(api, project_id)                  # the latest frozen dataset version
    policy = ss.policy(api, project_id)                 # the latest policy version
    run = ss.simulate(data, policy, replications=50, seed=42,
                      disruptions=[ss.outage("S2", start_day=140, duration_days=42)])
    ss.kpi_table(run["run"])

`simulate` returns the shapes the API returns — a run row, replication rows and
a long-form weekly series — so every helper (kpi_table, paired_compare,
plot_series, …) works on a local run exactly as on a platform run.
"""
from __future__ import annotations

import hashlib
import importlib.util
import subprocess
import sys
import tempfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Optional

import pandas as pd
import requests

# The Simulation Lab's defaults: 52 weeks (the engine's minimum), 10 replications,
# seed 42, common random numbers on, warm-up detected by the engine.
DEFAULT_FRAME = {"horizon_days": 364, "replications": 10, "seed": 42, "crn": True, "warmup_mode": "auto"}


# ── the engine ──────────────────────────────────────────────────────────────
def engine_available() -> bool:
    """Is the simulation engine importable in this environment?"""
    return importlib.util.find_spec("scsim") is not None and importlib.util.find_spec("sim_worker") is not None


def install_engine(api, quiet: bool = True, version: Optional[str] = None) -> str:
    """Download the engine through the API (a key with read:data), check every
    wheel's sha256, and pip-install it. Returns the engine version.

    ``version`` installs an EARLIER engine — ``"0.4.0"`` (the newest build of that
    version) or the exact build a run recorded, ``run["code_version"]``
    (``"scsim-0.4.0+<digest>"``) — so a stored result can be re-run on the engine
    that produced it. Omit it for the engine the platform runs now."""
    info = api.engine(version=version) if version else api.engine()
    with tempfile.TemporaryDirectory() as tmp:
        paths = []
        for w in info["wheels"]:
            body = requests.get(w["url"], timeout=120).content  # a signed URL: no API key sent
            digest = hashlib.sha256(body).hexdigest()
            if digest != w["sha256"]:
                raise RuntimeError(f"{w['file']}: sha256 {digest[:12]}… is not the published {w['sha256'][:12]}… — "
                                   "not installing it")
            path = Path(tmp) / w["file"]
            path.write_bytes(body)
            paths.append(str(path))
        cmd = [sys.executable, "-m", "pip", "install", "--upgrade", *(["-q"] if quiet else []), *paths]
        subprocess.check_call(cmd)
    importlib.invalidate_caches()
    build = info.get("engine_build") or "build not named"
    print(f"engine {info['engine_version']} ({build}) installed ({len(paths)} wheels, sha256 checked)")
    return info["engine_version"]


def _require_engine():
    if not engine_available():
        raise ImportError("the simulation engine is not installed here — run suresuite.install_engine(api) "
                          "(it needs an API key with read:data)")


# ── the frozen inputs ───────────────────────────────────────────────────────
@dataclass
class Dataset:
    """A frozen dataset version: its identity and its input tables."""
    id: str
    graph_hash: Optional[str]
    snapshot: dict
    label: Optional[str] = None
    version_no: Optional[int] = None
    meta: dict = field(default_factory=dict)

    @property
    def tables(self) -> dict[str, pd.DataFrame]:
        """The simulation's input tables, one DataFrame each."""
        inputs = self.snapshot.get("inputs") if isinstance(self.snapshot.get("inputs"), dict) else self.snapshot
        return {name: pd.DataFrame(rows) for name, rows in inputs.items() if isinstance(rows, list)}

    def __repr__(self) -> str:
        sizes = ", ".join(f"{k} {len(v)}" for k, v in self.tables.items() if len(v))
        return f"Dataset(v{self.version_no} {self.label!r}, graph_hash {str(self.graph_hash)[:12]}…: {sizes})"


@dataclass
class Policy:
    """A frozen policy version: its identity and its snapshot."""
    id: str
    policy_hash: Optional[str]
    snapshot: dict
    label: Optional[str] = None


def dataset(api, project_id: str, version: str = "latest") -> Dataset:
    """Pull a frozen dataset version (id or "latest") with its rows."""
    d = api.dataset_version(project_id, version)
    return Dataset(id=d["id"], graph_hash=d.get("graph_hash"), snapshot=d["snapshot"], label=d.get("label"),
                   version_no=d.get("version_no"), meta={k: v for k, v in d.items() if k != "snapshot"})


def policy(api, project_id: str, version: str = "latest") -> Policy:
    """Pull a frozen policy version (id or "latest")."""
    p = api.policy_version(project_id, version)
    return Policy(id=p["id"], policy_hash=p.get("policy_hash"), snapshot=p["snapshot"], label=p.get("label"))


def with_tables(data: Dataset, **tables: pd.DataFrame | list) -> Dataset:
    """A copy of `data` with some input tables replaced — a what-if on your own
    machine (more capacity, a second supplier, …). The platform's data is untouched."""
    snap = dict(data.snapshot)
    inputs = dict(snap.get("inputs") or {})
    for name, rows in tables.items():
        inputs[name] = rows.to_dict("records") if isinstance(rows, pd.DataFrame) else list(rows)
    snap["inputs"] = inputs
    return Dataset(id=f"{data.id} (edited locally)", graph_hash=None, snapshot=snap, label=f"{data.label} + local edits",
                   version_no=data.version_no, meta=dict(data.meta))


# ── the run ─────────────────────────────────────────────────────────────────
def simulate(data: Dataset | dict, policy_version: Policy | dict, scenario: Optional[dict] = None,
             disruptions: Optional[list] = None, project_model: Optional[str] = None,
             progress: bool = False, **frame: Any) -> dict:
    """Run the platform's engine on this machine. `scenario` is a scenario row (from
    `api.scenarios(...)`) or None; keyword arguments override its frame
    (horizon_days, replications, seed, crn, warmup_mode, …). Returns
    {"run", "replications", "series", "inputs"} in the API's shapes."""
    _require_engine()
    from sim_worker.local import run_from_snapshots

    snap = data.snapshot if isinstance(data, Dataset) else data
    pol = policy_version.snapshot if isinstance(policy_version, Policy) else policy_version
    sc = {**DEFAULT_FRAME, **{k: v for k, v in (scenario or {}).items()
                             if k in ("horizon_days", "warmup_days", "warmup_mode", "replications", "seed", "crn",
                                      "disruption_schedule", "stopping_rule", "demand_model", "recovery_overrides")},
          **frame}
    if disruptions is not None:
        sc["disruption_schedule"] = disruptions

    def _tick(_row, done, total):
        print(f"  replication {done}/{total}   ", end="\r")

    out = run_from_snapshots(snap, pol, sc, project_model=project_model, on_replication=_tick if progress else None)
    run = {**out["run_update"], "status": "done", "computed_by": "client", "engine_version": out["engine_version"],
           "graph_hash": getattr(data, "graph_hash", None), "policy_hash": getattr(policy_version, "policy_hash", None),
           "dataset_version_id": getattr(data, "id", None), "policy_version_id": getattr(policy_version, "id", None),
           "rep_count_target": sc["replications"]}
    if progress:
        print()
    return {"run": run, "replications": out["replications"], "series": pd.DataFrame(out["series"]),
            "inputs": {"scenario": sc}}

In [ ]:
# @title Study helpers (shared by notebooks 04 and 05)
# ── SuReSuite study helpers ─────────────────────────────────────────────────
# Shared by notebooks 04 and 05, so both read your project the same way: the data a
# run reads (frozen, or the newest frozen copy and its age), the policies as they are
# on /policies now (saved, or rebuilt here for a read-only key), your saved baseline
# scenario's frame, the platform's engine, and one replication's per-item weeks.
# Run this cell; you do not need to read it.
import json
import warnings
from pathlib import Path

import numpy as np

warnings.filterwarnings("ignore", category=FutureWarning)

INPUT_TABLES = ("suppliers", "materials", "products", "customers", "inbound", "outbound",
                "bom", "bom_multi_level", "demand_forecasts")


def key_scopes(api):
    """The scopes this key may use now (GET /key), so each step knows its fallback."""
    return set((api.key() or {}).get("scopes") or ["read:data", "read:policies"])


def key_table(scopes, verify_use="VERIFY_ON_PLATFORM"):
    return pd.DataFrame([
        {"needs": s, "this key": "yes" if s in scopes else "no", "used for": use, "without it": alt}
        for s, use, alt in [
            ("read:data", "your data and the engine", "—"),
            ("read:policies", "your policies", "—"),
            ("read:runs", "your saved baseline and stress-test scenarios", "the Simulation Lab's defaults"),
            ("write:data", "freezing the data as it is now", "the newest frozen copy (its age is shown)"),
            ("write:policies", "saving /policies as a version", "/policies read live and rebuilt here"),
            ("write:runs", verify_use, "skipped"),
        ]]).set_index("needs")


def styled(df, fmt=None, shade=(), cmap="Reds", caption=None):
    """A formatted table with value shading, or the plain table where the notebook cannot style."""
    try:
        s = df.style.format(fmt or {}, na_rep="—")
        for c in shade:
            if c in df and pd.api.types.is_numeric_dtype(df[c]) and df[c].notna().any():
                s = s.background_gradient(cmap=cmap, subset=[c])
        return s.set_caption(caption) if caption else s
    except (ImportError, AttributeError):  # pandas styling needs jinja2; without it `.style` raises AttributeError
        if caption:
            print(caption)
        return df


def capped(df, what="rows", limit=None):
    limit = limit or globals().get("MAX_ROWS", 40)
    if len(df) > limit:
        print(f"showing {limit} of {len(df)} {what} — the export has all of them")
        return df.head(limit)
    return df


def col(df, name, default=None):
    return df[name] if name in df else pd.Series(default, index=df.index)


def load_dataset(api, project_id, source, scopes, live, label):
    """The frozen dataset version a run reads. "current" freezes the data now when the key may
    (an unchanged project returns the version it has); without write:data it is the newest
    version, and its age is said. Versions are immutable, so they are cached on disk by id."""
    versions = sorted(api.dataset_versions(project_id), key=lambda v: v.get("created_at") or "", reverse=True)
    if source == "current" and "write:data" in scopes:
        version_id = api.freeze_dataset(project_id, label)["id"]
        print("froze the data as it is now:", "no change since the last freeze, so that version is used"
              if version_id in {v["id"] for v in versions} else "a new version")
    elif source in ("current", "latest"):
        if not versions:
            raise Stop("this project has no frozen data yet — run anything in the Simulation Lab (every run "
                       "freezes the data it reads) or use a key with write:data, then run again")
        version_id = versions[0]["id"]
        if source == "current":
            age = (pd.Timestamp.now(tz="UTC") - pd.Timestamp(versions[0]["created_at"])).days
            print(f"note: this key cannot freeze data (no write:data), so the newest frozen version is used — "
                  f"it is {age} day(s) old. Anything uploaded since is NOT in it. Every platform run freezes "
                  f"the data it reads: run anything in the Simulation Lab to refresh it.")
    else:
        version_id = source
    path = Path("suresuite_cache") / project_id / f"{version_id}.json"
    started = time.time()
    if path.exists():
        dv = json.loads(path.read_text())
        print(f"dataset version {version_id}: from the local cache")
    else:
        dv = api.dataset_version(project_id, version_id)
        if live:  # a version never changes, so a copy on disk is always current
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(json.dumps(dv))
        print(f"dataset version {version_id}: downloaded {len(json.dumps(dv)) / 1e6:.1f} MB "
              f"in {time.time() - started:.1f}s")
    return Dataset(id=dv["id"], graph_hash=dv.get("graph_hash"), snapshot=dv["snapshot"], label=dv.get("label"),
                   version_no=dv.get("version_no"), meta={k: v for k, v in dv.items() if k != "snapshot"})


def input_tables(data):
    return {name: data.tables.get(name, pd.DataFrame()) for name in INPUT_TABLES}


def flat_bom(T):
    """(product, material, rate): units of each purchased material per unit of each product,
    through any sub-assemblies. The multi-level BOM wins when it has rows, as in the engine."""
    ml = T["bom_multi_level"]
    if len(ml):
        children = {}
        for parent, child, rate in zip(col(ml, "higher_level_component_id"), ml["material_id"],
                                       col(ml, "consumption_rate", 1)):
            if parent:
                children.setdefault(str(parent), []).append((str(child), float(rate or 0)))
        out = {}

        def walk(product, node, mult, depth=0):
            for child, rate in children.get(node, []):
                if child in children and depth < 25:
                    walk(product, child, mult * rate, depth + 1)
                else:
                    out[(product, child)] = out.get((product, child), 0.0) + mult * rate
        for p in col(T["products"], "product_id").astype(str):
            walk(p, p, 1.0)
        return pd.DataFrame([{"product_id": p, "material_id": m, "rate": r} for (p, m), r in out.items()],
                            columns=["product_id", "material_id", "rate"])
    b = T["bom"]
    return pd.DataFrame({"product_id": col(b, "product_id").astype(str), "material_id": col(b, "material_id").astype(str),
                         "rate": col(b, "consumption_rate", 1).astype(float)})


PER_WEEK = {"day": 7, "week": 1, "month": 12 / 52, "quarter": 4 / 52, "year": 1 / 52}


def supply_lanes(T):
    """Inbound lanes with their weekly volume and weekly spend (for ranking only; the engine
    normalizes the data itself)."""
    return T["inbound"].assign(
        supplier_id=lambda d: col(d, "supplier_id").astype(str), material_id=lambda d: col(d, "material_id").astype(str),
        per_week=lambda d: col(d, "volume", 0).astype(float).fillna(0)
                           * col(d, "time_unit", "week").fillna("week").str.lower().map(PER_WEEK).fillna(1),
        spend=lambda d: d["per_week"] * col(d, "unit_price", 0).astype(float).fillna(0))


def data_check(T, bom, lanes):
    mats = set(col(T["materials"], "material_id").astype(str))
    prods = set(col(T["products"], "product_id").astype(str))
    n_sources = lanes.groupby("material_id")["supplier_id"].nunique()
    return pd.DataFrame([
        ("materials in the BOM with no item master", sorted(set(bom.material_id) - mats)),
        ("materials with no supplier lane", sorted(set(bom.material_id) - set(lanes.material_id))),
        ("products with no BOM", sorted(prods - set(bom.product_id))),
        ("materials with a single supplier", sorted(n_sources[n_sources == 1].index)),
    ], columns=["check", "items"]).assign(
        count=lambda d: d["items"].map(len),
        items=lambda d: d["items"].map(lambda xs: ", ".join(xs[:12]) + (" …" if len(xs) > 12 else "")))


def load_policies(api, project_id, source, pinned_id, scopes, label):
    """The policy version a run reads, and what differs between the saved version and /policies now.
    Returns (policy, drift rows, the families as stored on /policies now)."""
    saved = sorted(api.policy_versions(project_id), key=lambda v: v.get("created_at") or "", reverse=True)
    pinned_id = pinned_id or (saved[0]["id"] if saved else None)
    pinned = policy(api, project_id, pinned_id) if pinned_id else None

    live = api.policies(project_id)
    live_def = live.get("defaults") or {}
    now_fam = {f: dict(live_def.get(f) or {}) for f in FAMILIES}
    now_strategy = live_def.get("fulfillment_strategy")
    canon = lambda o: (o["scope"], o["target_key"], o["family"])
    now_over = {canon(o): o.get("patch") or {} for o in live.get("overrides") or []}

    drift = []
    if pinned:
        saved_def = pinned.snapshot.get("defaults") or {}
        for f in FAMILIES:
            for k, v in now_fam[f].items():
                if (saved_def.get(f) or {}).get(k) != v:
                    drift.append({"setting": f"{f}.{k}", "saved version": (saved_def.get(f) or {}).get(k), "/policies now": v})
        if now_strategy and now_strategy != pinned.snapshot.get("fulfillment_strategy"):
            drift.append({"setting": "fulfillment_strategy", "saved version": pinned.snapshot.get("fulfillment_strategy"),
                          "/policies now": now_strategy})
        saved_over = {canon(o): o.get("patch") or {} for o in pinned.snapshot.get("overrides") or []}
        for key in sorted(set(saved_over) | set(now_over)):
            if saved_over.get(key) != now_over.get(key):
                drift.append({"setting": "override " + " · ".join(key), "saved version": saved_over.get(key),
                              "/policies now": now_over.get(key)})
        print(f"saved version: {pinned.label or 'unlabelled'} ({pinned.id})")

    if source == "pinned" or (pinned and not drift):
        if not pinned:
            raise Stop("POLICY_SOURCE is \"pinned\" but this project has no saved policy version")
        pol = pinned
        print("using the saved version" + ("" if not drift else f" — {len(drift)} setting(s) differ from /policies now (below)"))
    elif "write:policies" in scopes:
        pol = policy(api, project_id, api.snapshot_policies(project_id, f"{label} · /policies as it is now")["id"])
        print(f"saved /policies as it is now: {pol.label} ({pol.id})")
    else:
        # The platform builds a version as the page's defaults with the stored values over them, family
        # by family. The saved version already holds the page's defaults, so the stored values go over it.
        snap = copy.deepcopy(pinned.snapshot) if pinned else {"schema_version": 2, "defaults": {}}
        for f in FAMILIES:
            snap.setdefault("defaults", {})[f] = {**(snap["defaults"].get(f) or {}), **now_fam[f]}
        snap["fulfillment_strategy"] = now_strategy or snap.get("fulfillment_strategy") or "make_to_stock"
        snap["overrides"] = [{"scope": s, "target_key": t, "family": f, "patch": p} for (s, t, f), p in sorted(now_over.items())]
        pol = Policy(id="rebuilt from /policies (not a saved version)", policy_hash=None, snapshot=snap,
                     label="/policies as it is now")
        print("note: this key cannot save a policy version (no write:policies), so the current policies are rebuilt "
              "here from /policies" + (" over the saved version." if pinned else
              " alone — there is no saved version, so settings never touched on /policies take the ENGINE's defaults, "
              "which can differ from the page's. Save a version on /policies for an exact run."))
    if not drift and pinned:
        print("the saved version matches /policies on every stored setting")
    return pol, pd.DataFrame(drift, columns=["setting", "saved version", "/policies now"]).set_index("setting"), now_fam


def policy_tables(pol, now_fam):
    """(project-wide settings with where each comes from, per-item overrides counted by family)."""
    snap = pol.snapshot or {}
    rows = [{"family": f, "setting": k, "value": v if not isinstance(v, (dict, list)) else json.dumps(v)[:80],
             "where": "set on /policies" if k in now_fam.get(f, {}) else "page default"}
            for f in FAMILIES for k, v in (snap.get("defaults", {}).get(f) or {}).items()]
    settings = pd.DataFrame(rows, columns=["family", "setting", "value", "where"]).sort_values(
        ["where", "family"], ascending=[False, True])
    over = pd.DataFrame(snap.get("overrides") or [], columns=["scope", "target_key", "family", "patch"])
    return settings, over.groupby(["family", "scope"]).size().rename("per-item overrides").to_frame()


def study_scenario(api, project_id, scenario_id, scopes, replications=None):
    """Your saved baseline scenario (or the Simulation Lab's defaults), its frame, and your saved
    stress tests. A pre-filled SCENARIO_ID with disruptions is used as a frame and its events
    join the stress tests."""
    scenarios = api.scenarios(project_id) if "read:runs" in scopes else []
    stress = [s for s in scenarios if s.get("disruption_schedule")]
    sc = next((s for s in scenarios if s["id"] == scenario_id), None) if scenario_id else \
        next((s for s in scenarios if not s.get("disruption_schedule")), None)
    if sc is None:
        sc = {"id": None, "name": "Simulation Lab defaults", **DEFAULT_FRAME}
        print("no saved baseline scenario" + ("" if "read:runs" in scopes else " (the key has no read:runs)")
              + " — using the Simulation Lab's defaults")
    elif sc.get("disruption_schedule"):
        print(f"note: {sc['name']} has disruptions — its frame is the baseline here, its events are a stress test")
        stress = [sc] + [s for s in stress if s["id"] != sc["id"]]
        sc = {**sc, "disruption_schedule": []}
    frame = {k: sc[k] for k in ("horizon_days", "warmup_days", "replications", "seed", "crn", "recovery_overrides")
             if sc.get(k) is not None}
    if replications:
        frame["replications"] = replications
    print(f"scenario: {sc['name']} — {frame.get('horizon_days')} days, {frame.get('replications')} replications, "
          f"seed {frame.get('seed')}")
    return sc, frame, stress


def ensure_engine(api, live):
    """The platform's current engine on this machine: installed, or reinstalled when its version
    is not the platform's. Demo mode hands out no engine."""
    try:
        platform_engine = api.engine()
    except SuReSuiteError:
        platform_engine = None
    if platform_engine and live:
        import importlib.metadata as md
        have = md.version("scsim") if engine_available() else None
        if have != platform_engine["engine_version"]:
            install_engine(api)
    if not engine_available():
        print("the engine is not installed — demo mode cannot hand it out. Add SURESUITE_API_KEY (see the Setup "
              "notes) and run all cells again; the simulation steps need it.")
        return False
    return True


def engine_view(data, pol, scenario, project_model):
    """What the engine RECEIVES for these inputs — lead times in weeks, capacities, opening stock,
    the allocation rule — read from the engine's own mapping rather than re-derived here."""
    from sim_worker.local import engine_input_from_snapshots
    return engine_input_from_snapshots(data.snapshot, pol.snapshot, scenario, project_model=project_model)["scenario"]


def inspect_run(data, pol, scenario, project_model, seed, disruptions=None):
    """One replication with every item's weekly record — {(kind, measure): weeks × items}. A one-replication
    run with seed s is replication 0 of any run with root seed s, so it is one of that run's own worlds."""
    from sim_worker.local import run_from_snapshots
    out = run_from_snapshots(data.snapshot, pol.snapshot,
                             {**scenario, "replications": 1, "seed": seed, "inspection": True,
                              "disruption_schedule": list(disruptions or [])},
                             project_model=project_model)
    frames = {}
    for r in out["item_series"]:
        for measure, values in r["series"].items():
            frames.setdefault((r["kind"], measure), {})[str(r["item_id"])] = values
    if not frames:
        raise Stop("the engine returned no per-item weeks — this engine build predates them; install the platform's")
    return out["run_update"], {k: pd.DataFrame(v) for k, v in frames.items()}


def use_of(frames, bom):
    """Each material used per week: each product's production times its BOM."""
    prod = frames.get(("product", "production"), pd.DataFrame())
    oh = frames[("material", "on_hand")]
    use = pd.DataFrame(0.0, index=oh.index, columns=oh.columns)
    for p, m, rate in bom.itertuples(index=False):
        if p in prod and m in use:
            use[m] += prod[p] * rate
    return use

In [ ]:
# @title Demo data (recorded engine output)
# ── DEMO DATA ───────────────────────────────────────────────────────────────
# Recorded engine output (scsim 0.8.2) for the Example project, replayed
# when no API key is found. Recorded by notebooks/tools/record_demo.py; not used in live mode.
# Recordings: baseline.
DEMO_DATA = "..."

## Try changing
- `DATA_SOURCE`, `POLICY_SOURCE` — `"current"` follows the project as it is now; pin a version id to repeat a study.
- `REPLICATIONS` — `None` uses your scenario's; fewer is faster, more gives tighter intervals.
- `MATERIALS_TO_TRACE` — material ids from step 2; `[]` picks the riskiest.
- `INSPECTION_SEED` — pin the one world the material traces show; `None` lets step 7 pick a typical one.
- `DISRUPTION_TARGETS`, `START_DAY`, `OUTAGE_DAYS`, `MAGNITUDE_PCT` — what to knock out, when, how long, how hard.
- `SUPPLIER_ID` (in the CONFIG cell) — the outage to follow in step 6; `""` = the worst one.

In [ ]:
DATA_SOURCE = "current"      # "current" | "latest" (newest frozen version) | a dataset version id
POLICY_SOURCE = "current"    # "current" = /policies as it is now | "pinned" = POLICY_VERSION_ID exactly
REPLICATIONS = None          # None = the scenario's replications
INSPECTION_SEED = None       # None = step 5 shows the baseline's first replication, step 7 the most typical seed
SEED_CANDIDATES = 6          # seeds step 7 tries when picking the most typical one
MATERIALS_TO_TRACE = []      # e.g. ["M2", "M6"]
N_AUTO_MATERIALS = 3
DISRUPTION_TARGETS = []      # supplier ids and/or "plant"; [] = the most exposed suppliers + the plant
MAX_AUTO_TARGETS = 10
INCLUDE_SAVED_STRESS_TESTS = True  # also run the project's own stress-test scenarios (Simulation Lab)
START_DAY, OUTAGE_DAYS, MAGNITUDE_PCT = 140, 42, 100
MAX_ROWS = 40                # long tables show this many rows; the export has all of them
VERIFY_ON_PLATFORM = False   # True = also run the baseline on the platform and compare (write:runs; uses quota)
EXPORT_XLSX = "component_inventory_study.xlsx"   # "" = no export

## 1 · Connect, and what this key can do
**Expect:** your project, then a table of what the key allows and what the notebook does without it.

In [ ]:
import matplotlib.colors as mcolors

api = connect(BASE_URL, API_KEY, DEMO_DATA, needs=["read:data", "read:policies"], mode=MODE)
LIVE = api.live
project = pick_project(api, PROJECT_ID)
PID = project["id"]
# The platform reads the project's model to decide how a product with no fulfillment mode of
# its own is made (make-to-stock or make-to-order). A local run must be told the same.
PROJECT_MODEL = project.get("supply_chain_model")
print(f"supply chain model: {PROJECT_MODEL or 'not set (make-to-order)'}")
SCOPES = key_scopes(api)
can = SCOPES.__contains__
display(key_table(SCOPES))

## 2 · Load the project's data
A run reads a **frozen dataset version** — your tables at one moment, identified by its
`graph_hash`. `"current"` freezes the data now when the key may (an unchanged project returns
the version it already has); otherwise it takes the newest version and says how old it is.
Versions are kept in `suresuite_cache/`, so running again does not download them again.
**Expect:** the version used, the size of every table, and a data check.

In [ ]:
data = load_dataset(api, PID, DATA_SOURCE, SCOPES, LIVE, "notebook 04")
print(data)
T = input_tables(data)
display(pd.DataFrame({"rows": {k: len(v) for k, v in T.items()}}).T)

**The bill of materials, flattened.** Units of each purchased material per unit of each product,
through any sub-assemblies (the multi-level BOM wins when it has rows, as in the engine).

In [ ]:
BOM = flat_bom(T)
lanes = supply_lanes(T)
mats = T["materials"].assign(material_id=lambda d: col(d, "material_id").astype(str)).set_index("material_id")
prods = T["products"].assign(product_id=lambda d: col(d, "product_id").astype(str)).set_index("product_id")
n_sources = lanes.groupby("material_id")["supplier_id"].nunique()
check = data_check(T, BOM, lanes)
display(check.set_index("check")[["count", "items"]])

## 3 · Load the policies as they are now
A run reads a **policy version**. `"current"` compares your saved version with /policies as it is
now: if they agree, the saved version is used (exact, with its `policy_hash`); if not, the key saves
a new version when it may (an unchanged configuration is never saved twice), and otherwise the
notebook rebuilds the current policies here from /policies and the saved version, and says so.
**Expect:** what differs between the saved version and /policies, and which one the runs use.

In [ ]:
pol, drift, now_fam = load_policies(api, PID, POLICY_SOURCE, POLICY_VERSION_ID, SCOPES, "notebook 04")
if len(drift):
    display(styled(capped(drift, "differences"), caption="Saved version vs /policies now"))
settings, overrides = policy_tables(pol, now_fam)
if len(settings):
    display(styled(capped(settings, "settings"),
                   caption=f"Project-wide settings the runs use — fulfillment strategy {pol.snapshot.get('fulfillment_strategy')}"))
if len(overrides):
    display(styled(overrides, caption=f"{int(overrides.iloc[:, 0].sum())} per-item overrides (they win over the project-wide settings)"))

## 4 · Run the baseline on this machine
The frame (horizon, warm-up, replications, seed, recovery playbook) is your saved baseline scenario's,
so the numbers are the Simulation Lab's. The engine is the platform's current build, installed through
the API once per environment.
**Expect:** the scenario used, the engine's notes on how it read your data, the KPIs, and four charts.

In [ ]:
scenario, FRAME, stress_saved = study_scenario(api, PID, SCENARIO_ID, SCOPES, REPLICATIONS)
RUN_LOCAL = ensure_engine(api, LIVE)


def simulate_here(disruptions=None, **over):
    """One run on this machine with the platform's inputs: this data, these policies, this frame,
    and the project's model."""
    return simulate(data, pol, scenario={**scenario, **FRAME, **over}, disruptions=disruptions or [],
                    project_model=PROJECT_MODEL)


if RUN_LOCAL:
    started = time.time()
    base = simulate_here()
    RUN_SECONDS = time.time() - started
    WARMUP = int(base["run"].get("warmup_detected_at") or 0)
    print(f"{FRAME.get('replications')} replications in {RUN_SECONDS:.1f}s · engine {base['run'].get('code_version')} · "
          f"warm-up ends week {WARMUP}")
    notes = pd.DataFrame(base["run"].get("mapping_warnings") or [], columns=["level", "entity", "field", "reason"])
    warn = notes[notes.level == "warn"]
    if len(warn):
        display(styled(capped(warn.groupby(["field", "reason"]).size().rename("items").sort_values(ascending=False)
                              .to_frame().reset_index(), "notes"),
                       caption=f"How the engine read your data — {len(warn)} warning(s)"))
    display(kpi_table(base["run"])[["measure", "mean", "± 95% CI"]])

In [ ]:
COLORS = {"base": "#2a78d6", "shock": "#d9480f", "transit": "#7a5af8", "orders": "#94a3b8", "use": "#0f766e",
          "fg": "#008300", "zero": "#e03131", "window": "#5f6b7a"}


def band(ax, df, column, color, label, scale=1.0):
    """Mean across replications with the 10th–90th percentile band."""
    g = df.groupby("week")[column]
    m = g.mean() * scale
    ax.plot(m.index, m.values, color=color, linewidth=1.8, label=label)
    ax.fill_between(m.index, g.quantile(0.1) * scale, g.quantile(0.9) * scale, color=color, alpha=0.15, linewidth=0)


def marks(ax, window=None, warmup=None, legend=True):
    if warmup:
        ax.axvline(warmup, color=COLORS["window"], linestyle=":", linewidth=1, label="warm-up ends" if legend else None)
    if window:
        ax.axvspan(*window, color=COLORS["window"], alpha=0.12, linewidth=0, label="disruption" if legend else None)
    ax.spines[["top", "right"]].set_visible(False)


if RUN_LOCAL:
    S = base["series"]
    fig, axes = plt.subplots(2, 2, figsize=(11, 6))
    band(axes[0, 0], S, "fill_rate", COLORS["base"], "fill rate", 100)
    axes[0, 0].set_title("Fill rate (%)", fontsize=10)
    band(axes[0, 1], S, "on_hand_value", COLORS["base"], "materials")
    band(axes[0, 1], S, "fg_value", COLORS["fg"], "finished goods")
    axes[0, 1].set_title("Inventory value", fontsize=10)
    band(axes[1, 0], S, "backlog_units", COLORS["shock"], "backlog")
    axes[1, 0].set_title("Backlog (units)", fontsize=10)
    if S.get("plant_capacity_units") is not None and S["plant_capacity_units"].fillna(0).gt(0).any():
        S = S.assign(plant=100 * S.plant_capacity_used_units / S.plant_capacity_units.replace(0, np.nan),
                     suppliers=100 * S.supplier_capacity_used_units / S.supplier_capacity_units.replace(0, np.nan))
        band(axes[1, 1], S, "plant", COLORS["fg"], "plant")
        band(axes[1, 1], S, "suppliers", COLORS["transit"], "suppliers")
    axes[1, 1].set_title("Capacity used (%)", fontsize=10)
    for ax in axes.flat:
        marks(ax, warmup=WARMUP, legend=ax is axes[0, 0])
        ax.set_xlabel("week")
        ax.legend(fontsize=8, frameon=False)
    fig.suptitle(f"Baseline — mean of {FRAME.get('replications')} replications, band 10th–90th percentile", fontsize=10)
    plt.tight_layout()
    plt.show()

## 5 · Every material, week by week
One more run with **one** replication and the engine's inspection trace on: each material's stock on
hand, in transit and orders placed, and each product's demand, production and deliveries. It is the
baseline's first replication, the same world as one line in step 4. Use is each product's production
times its BOM. Weeks before the warm-up ends are left out.
**Expect:** every material ranked by risk, then a chart per traced material.

In [ ]:
def inspect(seed, disruptions=None):
    return inspect_run(data, pol, {**scenario, **FRAME}, PROJECT_MODEL, seed, disruptions)


def material_table(frames, w0):
    post = lambda df: df[df.index >= w0]
    oh, use = post(frames[("material", "on_hand")]), post(use_of(frames, BOM))
    zero = oh <= 1e-6
    avg_use = use.mean().replace(0, np.nan)
    return pd.DataFrame({
        "suppliers": [", ".join(sorted(set(lanes.supplier_id[lanes.material_id == m]))) or "—" for m in oh.columns],
        "avg on hand": oh.mean(), "min on hand": oh.min(), "weeks at zero": zero.sum(),
        "first stockout": zero.apply(lambda c: c.idxmax() if c.any() else np.nan),
        "avg in transit": post(frames[("material", "in_transit")]).mean(), "avg use / wk": use.mean(),
        "min cover (wk)": oh.min() / avg_use, "ordered": post(frames[("material", "orders")]).sum(),
    }, index=oh.columns).rename_axis("material")


if RUN_LOCAL:
    SEED5 = INSPECTION_SEED if INSPECTION_SEED is not None else int(FRAME.get("seed", 42))
    insp_run, BASE = inspect(SEED5)
    W0 = int(insp_run.get("warmup_detected_at") or WARMUP)
    mt = material_table(BASE, W0)
    mt = mt.assign(_c=mt["min cover (wk)"].fillna(np.inf)).sort_values(["weeks at zero", "_c"], ascending=[False, True]).drop(columns="_c")
    display(styled(capped(mt, "materials"), shade=["weeks at zero"],
                   fmt={c: "{:,.1f}" for c in mt.columns if c not in ("suppliers", "weeks at zero", "first stockout")}
                   | {"weeks at zero": "{:,.0f}", "first stockout": "{:,.0f}"},
                   caption=f"Every material, riskiest first — seed {SEED5}, weeks {W0}+"))
    TRACE = [m for m in MATERIALS_TO_TRACE if m in mt.index] or list(mt.index[:N_AUTO_MATERIALS])
    if set(MATERIALS_TO_TRACE) - set(mt.index):
        print("not a material of this run (check the ids above):", sorted(set(MATERIALS_TO_TRACE) - set(mt.index)))
    print("tracing:", ", ".join(TRACE))

In [ ]:
def plot_material(m, frames, other=None, window=None, labels=("baseline", None)):
    oh, it, orders, use = (frames[("material", "on_hand")], frames[("material", "in_transit")],
                           frames[("material", "orders")], use_of(frames, BOM))
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 4.4), sharex=True, gridspec_kw={"height_ratios": [3, 2]})
    a1.plot(oh.index, oh[m], color=COLORS["base"], linewidth=2, label=f"on hand · {labels[0]}")
    a1.plot(it.index, it[m], color=COLORS["transit"], linewidth=1.2, linestyle="--", label=f"in transit · {labels[0]}")
    show = oh
    if other is not None:
        show = other[("material", "on_hand")]
        a1.plot(show.index, show[m], color=COLORS["shock"], linewidth=2, label=f"on hand · {labels[1]}")
        a1.plot(other[("material", "in_transit")].index, other[("material", "in_transit")][m], color=COLORS["shock"],
                linewidth=1.2, linestyle="--", alpha=0.7, label=f"in transit · {labels[1]}")
        orders, use = other[("material", "orders")], use_of(other, BOM)
    out = show.index[(show[m] <= 1e-6) & (show.index >= W0)]
    if len(out):
        a1.scatter(out, [0] * len(out), color=COLORS["zero"], marker="v", s=26, zorder=5, label="stockout")
    users = ", ".join(sorted(set(BOM.product_id[BOM.material_id == m]))) or "—"
    a1.set_title(f"{m} · {mats['name'].get(m, '') if 'name' in mats else ''} — from {mt.loc[m, 'suppliers']} · used by {users}",
                 fontsize=10)
    a1.set_ylabel("units")
    a2.bar(orders.index, orders[m], color=COLORS["orders"], width=0.8, label="ordered")
    a2.plot(use.index, use[m], color=COLORS["use"], linewidth=1.6, label="used")
    a2.set_ylabel("units / week")
    a2.set_xlabel("week")
    for ax in (a1, a2):
        marks(ax, window, W0, legend=ax is a1)
        ax.legend(fontsize=8, frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1.0))
    plt.tight_layout()
    plt.show()


if RUN_LOCAL:
    for m in TRACE:
        plot_material(m, BASE)
    mrp = [k for k in ("mrp_need", "mrp_on_hand", "mrp_on_the_way", "mrp_net") if ("material", k) in BASE]
    for m in TRACE if mrp else []:
        rec = pd.DataFrame({k: BASE[("material", k)].get(m) for k in mrp})
        if rec.notna().any().any() and rec.abs().to_numpy().sum() > 0:
            display(styled(rec[rec.index >= W0].head(12).T, fmt="{:,.0f}", caption=f"{m} — the MRP record, first 12 weeks"))

## 6 · Knock out each supplier
Each target ships nothing (or `MAGNITUDE_PCT` less) for `OUTAGE_DAYS` from `START_DAY`; deliveries are
deferred, not lost. Every run uses the baseline's random draws, so each is compared with it
replication by replication. Suppliers are chosen by **exposure**: what they ship that no one else does.
Your saved stress tests from the Simulation Lab run too. A target must be a supplier id or `plant`:
the engine skips any other, and the table says so.
**Expect:** the targets and the time it will take, then the impact of each, worst first.

In [ ]:
if RUN_LOCAL:
    sole = set(n_sources[n_sources == 1].index)
    exposure = (lanes.assign(sole=lanes.material_id.isin(sole))
                .groupby("supplier_id")
                .agg(materials=("material_id", lambda s: ", ".join(sorted(set(s)))),
                     sole_sourced=("sole", "sum"),
                     exposed_spend=("spend", lambda s: s[lanes.loc[s.index, "material_id"].isin(sole)].sum()),
                     spend=("spend", "sum"))
                .sort_values(["exposed_spend", "sole_sourced", "spend"], ascending=False))
    targets = list(DISRUPTION_TARGETS) or list(exposure.index[:MAX_AUTO_TARGETS]) + ["plant"]
    if SUPPLIER_ID and SUPPLIER_ID not in targets:
        targets.insert(0, SUPPLIER_ID)
    unknown = [t for t in targets if t != "plant" and t not in set(col(T["suppliers"], "supplier_id").astype(str))]
    if unknown:
        print("not a supplier id of this project — skipped:", unknown)
    cases = {f"{t} out": [outage(t, START_DAY, OUTAGE_DAYS, MAGNITUDE_PCT)] for t in targets if t not in unknown}
    if INCLUDE_SAVED_STRESS_TESTS:
        cases.update({f"saved: {s['name']}": s["disruption_schedule"] for s in stress_saved})
    display(styled(capped(exposure, "suppliers"), fmt={"exposed_spend": "{:,.0f}", "spend": "{:,.0f}"},
                   shade=["exposed_spend"], cmap="Oranges", caption="Suppliers by exposure (weekly spend)"))
    print(f"{len(cases)} runs × {FRAME.get('replications')} replications — about {len(cases) * RUN_SECONDS:.0f}s")

    shocks = {}
    for name, events in cases.items():
        started = time.time()
        r = simulate_here(disruptions=events)
        applied = any("recovery_measurable" in (x.get("kpis") or {}) for x in r["replications"])
        shocks[name] = {"events": events, "result": r, "applied": applied}
        print(f"  {name:<34} {time.time() - started:5.1f}s" + ("" if applied else "  — NOT APPLIED: the engine skipped every event"))

In [ ]:
IMPACT = [("fill_rate", "Δ fill rate (pp)", 100), ("lost_units", "Δ lost units", 1),
          ("lost_sales_value", "Δ lost sales", 1), ("max_backlog", "Δ peak backlog", 1),
          ("avg_on_hand_value", "Δ avg material stock value", 1)]


def paired_values(base_reps, reps, kpi):
    """(disrupted − baseline) for each replication, matched on the same random draws."""
    key = lambda r: (int((r.get("kpis") or {}).get("model_rep", r["rep_index"])), int((r.get("kpis") or {}).get("event_rep") or 0))
    a = {key(r): (r.get("kpis") or {}).get(kpi) for r in base_reps}
    return np.array([(r.get("kpis") or {}).get(kpi) - a[key(r)] for r in reps
                     if (r.get("kpis") or {}).get(kpi) is not None and a.get(key(r)) is not None], dtype=float)


def paired(base_reps, reps, kpi):
    """Mean and 95 % half-width of (disrupted − baseline), replication by replication."""
    d = paired_values(base_reps, reps, kpi)
    if len(d) < 2:
        return (d.mean() if len(d) else np.nan), np.nan
    return d.mean(), T95.get(len(d) - 1, 1.96) * d.std(ddof=1) / np.sqrt(len(d))


if RUN_LOCAL:
    rows = []
    for name, s in shocks.items():
        row = {"case": name, "applied": "yes" if s["applied"] else "NO"}
        for k, label, scale in IMPACT:
            m, hw = paired(base["replications"], s["result"]["replications"], k)
            row[label], row[label + " ±"] = m * scale, hw * scale
        reps = replications_frame(s["result"]["replications"])
        row["survives (wk)"] = reps["tts_weeks"].mean() if "tts_weeks" in reps else np.nan
        row["recovers (wk)"] = reps["ttr_weeks"].mean() if "ttr_weeks" in reps else np.nan
        m, hw = row["Δ lost sales"], row["Δ lost sales ±"]
        row["verdict"] = "no clear effect" if not abs(m) > (hw if hw == hw else 0) else ("loses sales" if m > 0 else "gains")
        rows.append(row)
    impact = pd.DataFrame(rows).set_index("case").sort_values("Δ lost sales", ascending=False)
    display(styled(capped(impact, "cases"), shade=["Δ lost sales", "Δ lost units"],
                   fmt={"Δ fill rate (pp)": "{:+.2f}", "Δ fill rate (pp) ±": "±{:.2f}", "Δ lost units": "{:+,.0f}",
                        "Δ lost units ±": "±{:,.0f}", "Δ lost sales": "{:+,.0f}", "Δ lost sales ±": "±{:,.0f}",
                        "Δ peak backlog": "{:+,.0f}", "Δ peak backlog ±": "±{:,.0f}",
                        "Δ avg material stock value": "{:+,.0f}", "Δ avg material stock value ±": "±{:,.0f}",
                        "survives (wk)": "{:.1f}", "recovers (wk)": "{:.1f}"},
                   caption=f"Impact vs baseline — mean of {FRAME.get('replications')} paired replications, ± 95 % interval"))
    print("survives = weeks the chain kept serving after the event began; recovers = weeks back to normal service")

    shown = impact.iloc[:20][::-1]
    fig, axes = plt.subplots(1, 2, figsize=(11, 0.36 * len(shown) + 1.4))
    for ax, label in zip(axes, ["Δ lost sales", "Δ fill rate (pp)"]):
        vals, err = shown[label], shown[label + " ±"].fillna(0)
        ax.barh(range(len(shown)), vals, xerr=err, error_kw={"linewidth": 0.8, "capsize": 2},
                color=[COLORS["shock"] if abs(v) > e else "#adb5bd" for v, e in zip(vals, err)])
        ax.set_yticks(range(len(shown)), shown.index, fontsize=8)
        ax.axvline(0, color="#495057", linewidth=0.8)
        ax.set_title(label, fontsize=10)
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle("Worst first — grey: not separable from noise at 95 %", fontsize=10)
    plt.tight_layout()
    plt.show()

## 7 · Follow one outage to the materials
The outage in `SUPPLIER_ID`, or else the worst supplier outage, runs again as one replication, with and
without the outage on the **same seed**, so the two worlds share every random draw and differ only by
the outage. An outage often hurts in some worlds and not at all in others (it depends on where it
falls in the reorder cycle), so the notebook says how often it hurt, tries `SEED_CANDIDATES` seeds, and
follows a world where it hurt about as much as it usually does when it hurts.
**Expect:** the seed chosen and why, a map of how every material's stock moved, a table of the
materials touched, a chart for each, and what each product lost.

In [ ]:
if RUN_LOCAL:
    applied = [c for c in impact.index if shocks[c]["applied"]]
    FOCUS = f"{SUPPLIER_ID} out" if f"{SUPPLIER_ID} out" in shocks else \
        next((c for c in applied if c.endswith(" out") and c != "plant out"), applied[0] if applied else None)
    if FOCUS is None:
        raise Stop("no disruption reached the engine — check the targets in step 6")
    events = shocks[FOCUS]["events"]
    WINDOW = effective_window(events[0], base["run"])
    hit_supplier = events[0]["target"] if len(events) == 1 else None
    lost = lambda frames: frames[("product", "lost_units")].loc[W0:].to_numpy().sum() if ("product", "lost_units") in frames else 0.0
    rep_d = paired_values(base["replications"], shocks[FOCUS]["result"]["replications"], "lost_units")
    hurt_d = rep_d[rep_d > 0.5]
    typical = float(np.median(hurt_d)) if len(hurt_d) else 0.0
    print(f"{FOCUS} cost sales in {len(hurt_d)} of {len(rep_d)} replications"
          + (f" — when it did, a median {typical:,.0f} units" if len(hurt_d) else ""))
    seeds = [INSPECTION_SEED] if INSPECTION_SEED is not None else [int(FRAME.get("seed", 42)) + i for i in range(SEED_CANDIDATES)]
    tried = {}
    for sd in seeds:
        b = BASE if sd == SEED5 else inspect(sd)[1]
        o = inspect(sd, events)[1]
        tried[sd] = (b, o, lost(o) - lost(b))
    hurting = {sd: t for sd, t in tried.items() if t[2] > 0.5} if len(hurt_d) else {}
    SEED7 = min(hurting or tried, key=lambda sd: abs(tried[sd][2] - typical))
    if len(hurt_d) and not hurting:
        print(f"none of the {len(tried)} seeds tried lost sales — raise SEED_CANDIDATES to find one that does")
    BASE7, SHOCK, _ = tried[SEED7]
    if len(tried) > 1:
        display(styled(pd.DataFrame({"Δ lost units": {sd: t[2] for sd, t in tried.items()}}).rename_axis("seed").T,
                       fmt="{:+,.0f}", caption=f"{FOCUS}, one world per seed"))
    print(f"following: {FOCUS} — weeks {WINDOW[0]}–{WINDOW[1]}, seed {SEED7}, where it loses {tried[SEED7][2]:+,.0f} units")

    oh_b, oh_s = BASE7[("material", "on_hand")], SHOCK[("material", "on_hand")]
    rel = (oh_s - oh_b) / oh_b[oh_b.index >= W0].mean().replace(0, np.nan)
    rel = rel.loc[:, rel.abs().max().sort_values(ascending=False).index]
    rel = rel.loc[:, rel.abs().max() > 0.01].iloc[:, :MAX_ROWS]
    if rel.shape[1]:
        lim = float(min(1.0, np.nanmax(np.abs(rel.to_numpy()))))
        fig, ax = plt.subplots(figsize=(11, 0.3 * rel.shape[1] + 1.4))
        im = ax.imshow(rel.T.to_numpy(), aspect="auto", cmap="RdBu", norm=mcolors.TwoSlopeNorm(0, -lim, lim),
                       extent=[-0.5, rel.index.max() + 0.5, rel.shape[1] - 0.5, -0.5])
        ax.set_yticks(range(rel.shape[1]), rel.columns, fontsize=8)
        for x in WINDOW:
            ax.axvline(x, color="#212529", linewidth=0.8, linestyle="--")
        ax.set_xlabel("week")
        ax.set_title(f"Stock with {FOCUS} minus baseline, as a share of the baseline average (red = less)", fontsize=10)
        fig.colorbar(im, ax=ax, shrink=0.8, format=lambda v, _: f"{v:+.0%}")
        plt.tight_layout()
        plt.show()
    else:
        print("no material's stock moved by more than 1 % in this seed")

In [ ]:
def back_in_range(m, slack=0.05):
    """The week from which stock stays inside the baseline's own range (its min to max after warm-up).
    A reorder cycle shifted in time stays inside it, which a week-by-week comparison would not."""
    b = oh_b.loc[oh_b.index >= W0, m]
    s = oh_s.loc[oh_s.index >= WINDOW[0], m]
    outside = s[(s < b.min() * (1 - slack)) | (s > b.max() * (1 + slack))]
    if outside.empty:
        return "never left"
    return "not by the end" if outside.index.max() >= oh_s.index.max() else int(outside.index.max()) + 1


if RUN_LOCAL:
    mb, ms = material_table(BASE7, W0), material_table(SHOCK, W0)
    direct = set(lanes.material_id[lanes.supplier_id == hit_supplier]) if hit_supplier else set()
    touched = pd.DataFrame({
        "hit directly": ["yes" if m in direct else "" for m in mb.index], "suppliers": mb["suppliers"],
        "min on hand · base": mb["min on hand"], "min on hand · outage": ms["min on hand"],
        "weeks at zero · base": mb["weeks at zero"], "weeks at zero · outage": ms["weeks at zero"],
        "first stockout": ms["first stockout"], "Δ avg on hand": ms["avg on hand"] - mb["avg on hand"],
        "Δ used": (ms["avg use / wk"] - mb["avg use / wk"]) * (len(oh_b) - W0),
        "back in normal range": [back_in_range(m) for m in mb.index],
    }, index=mb.index)
    moved = (touched["Δ avg on hand"].abs() > 1e-6) | (touched["weeks at zero · outage"] != touched["weeks at zero · base"])
    touched = touched[moved | touched.index.isin(direct)].sort_values(["hit directly", "weeks at zero · outage"], ascending=False)
    display(styled(capped(touched, "materials"), shade=["weeks at zero · outage"],
                   fmt={c: "{:,.0f}" for c in ("min on hand · base", "min on hand · outage", "weeks at zero · base",
                                               "weeks at zero · outage", "first stockout")}
                   | {"Δ avg on hand": "{:+,.0f}", "Δ used": "{:+,.0f}"},
                   caption=f"Materials touched by {FOCUS} — seed {SEED7}, weeks {W0}+"))
    print("normal range = the material's own baseline min..max after warm-up.\n"
          "Δ used below zero on a material the outage did not hit: production stopped for want of ANOTHER material.")
    for m in list(dict.fromkeys([*touched.index[:4], *[t for t in TRACE if t in touched.index]])):
        plot_material(m, BASE7, SHOCK, WINDOW, labels=("baseline", FOCUS))

In [ ]:
def product_table(frames, w0):
    post = lambda k: frames.get(("product", k), pd.DataFrame())
    dem, ful = post("demand"), post("fulfillment")
    t = pd.DataFrame({"demand": dem[dem.index >= w0].sum(), "delivered": ful[ful.index >= w0].sum(),
                      "lost units": post("lost_units")[lambda d: d.index >= w0].sum(),
                      "peak backlog": post("backlog")[lambda d: d.index >= w0].max()})
    t["fill rate %"] = 100 * t["delivered"] / t["demand"].replace(0, np.nan)
    return t.rename_axis("product")


if RUN_LOCAL:
    pb, ps = product_table(BASE7, W0), product_table(SHOCK, W0)
    hurt = pd.DataFrame({
        "materials hit": [", ".join(sorted(set(BOM.material_id[BOM.product_id == p]) & direct)) for p in pb.index],
        "fill rate % · base": pb["fill rate %"], "fill rate % · outage": ps["fill rate %"],
        "Δ delivered": ps["delivered"] - pb["delivered"], "Δ lost units": ps["lost units"] - pb["lost units"],
        "Δ peak backlog": ps["peak backlog"] - pb["peak backlog"],
    }).sort_values("Δ lost units", ascending=False)
    display(styled(capped(hurt, "products"), shade=["Δ lost units"],
                   fmt={"fill rate % · base": "{:.1f}", "fill rate % · outage": "{:.1f}", "Δ delivered": "{:+,.0f}",
                        "Δ lost units": "{:+,.0f}", "Δ peak backlog": "{:+,.0f}"},
                   caption=f"Products under {FOCUS} — seed {SEED7}"))
    worst = list(hurt.index[:4])
    fig, axes = plt.subplots(1, len(worst), figsize=(min(3.4 * len(worst), 13), 2.8), squeeze=False)
    for ax, p in zip(axes[0], worst):
        ax.plot(SHOCK[("product", "demand")][p], color=COLORS["orders"], linewidth=1.2, label="demand")
        ax.plot(BASE7[("product", "fulfillment")][p], color=COLORS["base"], linewidth=1.6, label="delivered · base")
        ax.plot(SHOCK[("product", "fulfillment")][p], color=COLORS["shock"], linewidth=1.6, label="delivered · outage")
        marks(ax, WINDOW, W0, legend=False)
        ax.set_title(p, fontsize=10)
        ax.set_xlabel("week")
    axes[0, 0].legend(fontsize=8, frameon=False)
    plt.tight_layout()
    plt.show()

**Reading it.** Start at the material that ran out first and read backwards. Stock falling while orders
stay flat: the policy is not reacting. Orders rising while nothing arrives: the supply is not answering.
A material the outage did not hit, used less: production stopped for want of another one, and its
stock piles up. To tell the chain from the seed, set `INSPECTION_SEED` to another seed and run steps 5
and 7 again: a stockout that stays in the same week is structural.

## 8 · Export, and what produced these numbers
Anyone with access to the project gets the same numbers from the same versions and seeds.
**Expect:** the record, and the workbook written (in Colab, downloaded).

In [ ]:
if RUN_LOCAL:
    record = pd.Series({
        "project": f"{project['name']} ({PID})", "supply chain model": PROJECT_MODEL,
        "dataset version": f"{data.id} · v{data.version_no} · graph_hash {data.graph_hash}",
        "policy version": f"{pol.label} ({pol.id}) · policy_hash {pol.policy_hash}",
        "scenario": f"{scenario['name']} ({scenario.get('id')})",
        "frame": f"{FRAME.get('horizon_days')} days · {FRAME.get('replications')} replications · seed "
                 f"{FRAME.get('seed')} · warm-up week {WARMUP}",
        "material-trace seeds": f"step 5: {SEED5} · step 7: {SEED7}", "engine": base["run"].get("code_version"),
        "computed by": "this machine (client)",
    }, name="value")
    display(record.to_frame())
    if EXPORT_XLSX:
        long = pd.concat([df.rename_axis("week").reset_index().melt(id_vars="week", var_name="item", value_name="value")
                          .assign(world=world, kind=kind, measure=measure)
                          for world, frames in ((f"baseline · seed {SEED5}", BASE), (f"baseline · seed {SEED7}", BASE7),
                                                (f"{FOCUS} · seed {SEED7}", SHOCK)) for (kind, measure), df in frames.items()])
        sheet = lambda s: "".join("_" if ch in "[]:*?/\\" else ch for ch in s)[:31]
        with pd.ExcelWriter(EXPORT_XLSX) as xw:
            record.to_frame().to_excel(xw, sheet_name="record")
            check.to_excel(xw, sheet_name="data check", index=False)
            kpi_table(base["run"]).to_excel(xw, sheet_name="baseline KPIs")
            mt.to_excel(xw, sheet_name="materials (baseline)")
            impact.to_excel(xw, sheet_name="disruption impact")
            touched.to_excel(xw, sheet_name=sheet(f"materials · {FOCUS}"))
            hurt.to_excel(xw, sheet_name=sheet(f"products · {FOCUS}"))
        long.to_csv(EXPORT_XLSX.rsplit(".", 1)[0] + "_weekly_items.csv", index=False)
        print("written:", EXPORT_XLSX, "and", EXPORT_XLSX.rsplit(".", 1)[0] + "_weekly_items.csv")
        try:
            from google.colab import files
            files.download(EXPORT_XLSX)
        except ImportError:
            pass

## 9 · (Optional) The same baseline on the platform
With `VERIFY_ON_PLATFORM = True` and a key with `write:runs`, the platform runs the baseline too and the
two are compared KPI by KPI. It needs a saved policy version and a saved scenario, and uses your quota.
**Expect:** `identical on every KPI: True`.

In [ ]:
if VERIFY_ON_PLATFORM and RUN_LOCAL:
    if not (can("write:runs") and can("read:runs")):
        print("skipped: the key has no write:runs")
    elif pol.policy_hash is None or scenario.get("id") is None:
        print("skipped: the platform runs a SAVED policy version and a SAVED scenario — this study used "
              + ("policies rebuilt from /policies" if pol.policy_hash is None else "the Simulation Lab's defaults"))
    elif REPLICATIONS and REPLICATIONS != scenario.get("replications"):
        print("skipped: REPLICATIONS differs from the scenario's, so the two runs would not be the same run")
    else:
        platform = api.run_and_wait(PID, scenario["id"], pol.id, acknowledge_warnings=ACKNOWLEDGE_WARNINGS)
        keys = [k for k, v in (platform.get("aggregate_kpis") or {}).items() if isinstance(v, (int, float)) and not k.startswith("_")]
        same = all(platform["aggregate_kpis"][k] == base["run"]["aggregate_kpis"].get(k) for k in keys)
        print("identical on every KPI:", same)
        if not same:
            print("platform data", platform.get("dataset_version_id"), "vs this study's", data.id,
                  "· engines", platform.get("code_version"), "vs", base["run"].get("code_version"))
        display(pd.DataFrame({"platform": {kpi_label(k): fmt_kpi(k, platform["aggregate_kpis"][k]) for k in keys},
                              "this machine": {kpi_label(k): fmt_kpi(k, base["run"]["aggregate_kpis"].get(k)) for k in keys}}))

## Next
- **02** — the same outages as platform runs, saved where the Simulation Lab shows them.
- **03** — fix what you found on a local copy first: a second supplier lane, more safety stock.